# Deposit Attrition EDA — v11 · what drives the score, and why the client is leaving

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v10d settled the operating definition. From here on there is **one model**: `all_features` (148 columns),
`A_full_exit`, H = 6, IPW + damped Newton + weighted isotonic, trained and scored only where the money is
still here, ranked on p × normal balance^0.5, 1,000 alerts a month. The deposit-only / payment-only / both
comparisons are closed and are not re-run here.

This notebook answers the two questions an RM asks when a name lands on their list:

| # | Question | Section |
|---|---|---|
| 1 | **Why is this client on my list?** Which signals carry the score — ranked, with their direction, and which ones earn nothing | §2–§4 |
| 2 | **Is the client leaving because the business is shrinking, or because it is moving its banking elsewhere?** Contraction calls for credit, working capital, restructuring; displacement calls for competing on price, ECR and service | §5–§9 |

### Question 1 — ranking predictive power honestly
A coefficient in a 148-column ridge model is not an importance: correlated signals split or trade weight.
So every signal (an `ld_` value and its `md_` missingness flag, treated as one unit) is measured through
three lenses, on the same seven rolling origins and the money-still-here risk set:

- **Standalone** — AUC of a model holding that signal alone. *Does it predict on its own?*
- **Unique contribution** — AUC lost when the signal is removed and the model is **refitted** without it.
  *Does the model need it, or can the others cover for it?*
- **Reliance** — AUC lost when the signal is shuffled in the test month, the fitted model unchanged.
  *How much does this model lean on it?*

The consensus rank averages the three. A signal that is strong alone but adds nothing uniquely is
**redundant** — informative for a narrative, safe to drop from the model. Families are dropped whole and
judged on the ship metric (capped dollars reached, paired bootstrap). §4 turns the fitted model into
**reason codes** — the top three signals behind each alert — which is what an RM actually reads.

### Question 2 — why v8 failed, and the fix
v8 asked *"is the client's counterparty still alive?"* Almost every counterparty is still paid by someone,
so the axis was degenerate (stayer median 1.000). That was the wrong question. The one that separates the
two stories is **"is the client's business still trading — anywhere we can see?"** Three kinds of direct
evidence answer it, all from the payment table:

| Evidence | Displacement (moving banks) | Contraction (business shrinking) |
|---|---|---|
| **Self-directed transfers** — outbound to the client's own name at a non-PNC institution, above its own baseline | money moved to itself elsewhere | absent |
| **Relationship survival** — the client's former PNC trading partners, after exit: do they keep paying (or being paid by) the client *at another bank*? | relationships continue elsewhere | relationships stop |
| **A dominant new institution** — one bank the client never used before takes a large share of the drain | the new bank | *(ambiguous — also what a sale or wind-down looks like; kept as its own class)* |
| **Internal move** — the money goes to a sibling entity in the same `rltn_pwr_id`, or a same-named PNC customer | not attrition at all — a re-key | |

Relationship survival is only visible where the partner is itself a PNC customer — that coverage is
measured, not assumed. Name matching can miss, and a miss looks exactly like "relationships stopped", so a
**recall proxy** is measured and gates the contraction call. Every threshold is calibrated against
**stayers at pseudo-event months**, so each indicator fires for at most 5% of clients who did not leave.

The direct evidence is mostly visible only *around and after* the exit — too late for the RM. So the
directly-typed clients become labels, and §8 asks whether the type can be **predicted at alert time**
(rel_m −6 … −1, money still here) from the 148 model features plus evidence-to-date. §9 puts
p(leave) and p(displacement | leave) side by side on the recommended queue and maps them to a play.

> **Compliance.** Destination institutions are used only as "new or not" — no institution is named in any
> table. Names are matched but never displayed (`SHOW_IDS = False`; full ids go to CSVs only). The
> permissible-use read on destination data and a fair-lending review of any credit-offer routing are
> preconditions before this reaches an RM (`08` item 10).

### Runtime
§1–§4 ~15 min (the drop-one refits dominate; cached and resumable). **§5a is the long one: one Spark job per
payment month, ~2–5 min each, ~2–3 h the first time**; every month is written to HDFS and skipped on rerun.
Charts are saved to `OUT_DIR`, never shown; tables stay inline.

### Pre-registered predictions — scored mechanically in §10

| # | prediction | confidence |
|---|---|---|
| P1 | The refit reproduces v10c: mean money-still-here AUC within ±0.003 of **0.7483** | ~85% |
| P2 | The top 10 signals (ranked on the first four folds) stay within **0.010 AUC** of all 74 on the last three | ~60% |
| P3 | The v7 base payment family is the costliest family to drop | ~75% |
| P4 | At least **a third** of signals add nothing uniquely (mean drop-one ΔAUC ≤ 0) | ~65% |
| P5 | Standalone and unique-contribution rankings disagree: Spearman **< 0.6** | ~60% |
| P6 | Direct displacement evidence fires for **≥ 2×** as many A attriters as stayers | ~70% |
| P7 | Among directly typed A attriters, displacement holds **≥ 60%** of the normal balance | ~60% |
| P8 | Relationship survival is observable for **≥ 40%** of typeable A attriters | ~50% |
| P9 | The name-matching recall proxy is **≥ 50%** | ~50% |
| P10 | Drain speed separates the directly typed groups in the predicted direction, AUC **≥ 0.60** (displacement drains faster) | ~55% |
| P11 | At alert time, p(displacement \| leaving) reaches grouped-CV AUC **≥ 0.70** | ~50% |
| P12 | Internal moves are **< 5%** of typeable A attriters | ~70% |

In [ ]:
# =====================================================================
# 0 · CONFIGURATION AND HELPERS — v11
# =====================================================================
# Pure python / numpy / pandas. Spark is imported only in §1 and §5, so
# every analytical function here can be tested without a cluster.
import warnings, time, math, hashlib, calendar
import numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display, HTML
# Only library deprecation chatter is silenced. RuntimeWarnings stay visible:
# a blanket filter hid a real divide bug in the graph work (09 §3).
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── paths ─────────────────────────────────────────────────────────────
# TRAP: pathlib collapses hdfs://host/p -> hdfs:/host/p. HDFS paths stay strings.
HDFS_ROOT = globals().get("HDFS_ROOT_OVERRIDE", "hdfs://nameservice1/user/pk36814")
def hp(ver, name): return f"{HDFS_ROOT.rstrip('/')}/attrition_{ver}/{name}"
PAY_TBL = globals().get("PAY_TBL_OVERRIDE", "dsihd01p_dsi.neo4j_payments")
DEP_TBL = globals().get("DEP_TBL_OVERRIDE", "dsihd01p_dsi.lap_dsi_universe_optimized")
DATE_START, DATE_END = "2024-01-01", "2026-07-31"
OUT_DIR = Path(globals().get("OUT_DIR_OVERRIDE",
                             "/projects/DSI/sa15474/repos/pkg/eda/attrition_v11"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── the operating definition (00_START_HERE) — fixed, not tuned here ──
DEFN, CFG, FS = "A_full_exit", "defend", "all_features"
SEED, MAX_ROWS = 20260909, 60
ORIGIN_START_OFF = 18          # OFFSET from M_MIN — m_idx is ABSOLUTE (~24289-24319)
PRIMARY_H = 6
NEG_SAMPLE = 0.15
L2, MIN_TRAIN_POS, CAL_MONTHS = 2.0, 200, 2
DEFEND_FRAC, DEFEND_MIN = 0.50, 10_000.0      # "money still here"
DRAINED_FRAC = 0.10
QUEUE_K, ALPHA = 1000, 0.5                    # alpha fixed in advance (v10d)
RM_COST_PER_CALL = 250.0
N_BOOT, JACK_K = 4000, [0, 1, 2, 3, 5, 10, 20]
# what the refit must reproduce: v10c retrained AUC; v10d §6b capped headline
REF_AUC_DEFEND, REF_TP_CLIENTS, REF_DOLLARS = 0.7483, 229, 606.8e6

# ── §3 ranking ────────────────────────────────────────────────────────
RUN_DROP_ONE = True            # ~74 signals x 7 folds refits; cached + resumable
PERM_REPS = 3
DROP_MIN = 0.0005              # mean ΔAUC a "core" signal must clear ...
CORE_FOLDS = 6                 # ... and be positive in at least this many folds
REDUNDANT_MIN = 0.05           # |standalone AUC - 0.5| that counts as "predictive alone"
TOPN_GRID = [5, 10, 20, 40]
RANK_SPLIT = 4                 # rank on the first 4 folds; test reduced models on the rest

# ── §5 evidence of displacement vs contraction ────────────────────────
RUN_EVIDENCE_BUILD = True
REBUILD_EVIDENCE = False       # True re-runs months already on HDFS
KEY_LEN, MIN_KEY_CHARS = 12, 8 # compact name key: first 12 alnum chars, >= 8 to be usable
DEST_TOP_N, INT_TOP_N, SH_TOP_N = 10, 50, 200
SHADOW_RAILS = ["ACH", "WIRE", "RTP_PRT", "RTP_P2P", "CHECK"]   # card descriptors excluded
BASE_WIN, DRAIN_WIN, POST_WIN = (-12, -9), (-8, -1), (0, 5)     # rel_m windows
KNOWN_DEST_END = -9            # an institution used at or before rel_m -9 is not "new"
R_WIN, F_WIN = (-2, 0), (-8, -5)   # evidence-to-date at month t: recent vs reference
NEW_BLANK_M = 4                # "new institution" undefined in the first 4 panel months
TYPE_MIN_NORM = 10_000.0       # typing population: normal balance >= $10k at rel_m -12
STAYER_Q = 0.95                # each indicator fires for <= 5% of stayers at pseudo-events
FLOOR_SELF, FLOOR_NEWBANK, FLOOR_CONT, FLOOR_SIB = 0.25, 0.25, 0.10, 0.50
CEIL_DEAD = 0.10
ELIG_MIN_AMT, ELIG_MIN_PARTNERS, POST_MIN_M = 1_000.0, 2, 3
RECALL_MIN, RECALL_MIN_N = 0.50, 20
P_COMPETE, P_SUPPORT = 0.65, 0.35
TYPE_FOLDS, MIN_TYPE_N = 5, 50
RM_SAMPLE_PER_TYPE = 20
SHOW_IDS = False               # tables show a short hash; full ids go to CSVs only
RES, SAVED = {}, []

HAVE_MPL = True
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
    plt.rcParams.update({
        "figure.dpi": 120, "font.size": 9, "axes.spines.top": False,
        "axes.spines.right": False, "axes.grid": True, "grid.color": "#E5E7EB",
        "grid.linewidth": .7, "axes.edgecolor": "#9AA1AC",
        "figure.facecolor": "white", "axes.facecolor": "white"})
except Exception as e:
    HAVE_MPL = False
    print(f"  matplotlib unavailable ({e}) — no charts will be saved")
ACC, ACC2, GOOD, WARN, GREY, INK = "#C1440E", "#4A6FA5", "#2F6F4E", "#B8860B", "#9AA1AC", "#16181D"
FAMCOL = {"deposit": GREY, "payment_v7": ACC2, "fin_in": ACC, "cptya_out": GOOD,
          "fin_out2": WARN, "recurring": "#7B5EA7"}
TYPECOL = {"DISPLACEMENT": ACC, "MOVED_TO_NEW_BANK": "#E08E5B", "CONTRACTION": ACC2,
           "LEANS_CONTRACTION": "#8FA8CC", "MIXED": WARN, "INTERNAL": GOOD,
           "NO_DIRECT_EVIDENCE": GREY, "NOT_TYPEABLE": "#D1D5DB"}

# ── display ───────────────────────────────────────────────────────────
def usd(v):
    try: v = float(v)
    except (TypeError, ValueError): return "—"
    if not np.isfinite(v): return "—"
    a, sg = abs(v), ("−" if v < 0 else "")
    if a >= 1e9: return f"{sg}${a/1e9:,.2f}bn"
    if a >= 1e6: return f"{sg}${a/1e6:,.1f}m"
    if a >= 1e3: return f"{sg}${a/1e3:,.0f}k"
    return f"{sg}${a:,.0f}"
def pctf(v, d=1):
    try:
        v = float(v)
        return f"{v:.{d}%}" if np.isfinite(v) else "—"
    except (TypeError, ValueError): return "—"
def f4(v):
    try:
        v = float(v)
        return f"{v:+.4f}" if np.isfinite(v) else "—"
    except (TypeError, ValueError): return "—"
def cid(x):
    return str(x) if SHOW_IDS else "c_" + hashlib.sha1(str(x).encode()).hexdigest()[:6]
def _dec(s):
    from pyspark.sql import functions as F
    o = s
    for c, t in s.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o
def disp(o, title=None, n=None, save=None):
    n = MAX_ROWS if n is None else n
    out = o.copy() if isinstance(o, pd.DataFrame) else pd.DataFrame(o)
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}</div>"))
    display(out.head(n)); return out
def kv(pairs, title=None, save=None):
    items = list(pairs.items()) if isinstance(pairs, dict) else list(pairs)
    labs = [k for k, _ in items]
    d = sorted({k for k in labs if labs.count(k) > 1})
    if d: raise ValueError(f"kv(): duplicate labels {d}")
    return disp(pd.DataFrame({"metric": labs, "value": [str(v) for _, v in items]}),
                title=title, n=len(items), save=save)
def sdiv(a, b):
    """a / b where b > 0, NaN elsewhere — without evaluating the division on
    the masked elements (np.where(c, a/b, x) still divides everywhere)."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    out = np.full(np.broadcast(a, b).shape, np.nan)
    ok = np.isfinite(b) & (b > 0)
    np.divide(a, b, out=out, where=ok)
    return out

def save_frame(df, name):
    """parquet where pyarrow is available, CSV otherwise — a missing engine must
    not stop a three-hour run at the last step."""
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False); return f"{name}.parquet"
    except Exception:
        df.to_csv(OUT_DIR / f"{name}.csv", index=False); return f"{name}.csv"

# ── charts: SAVED ONLY, never displayed ───────────────────────────────
def _save(fig, name, title, sub=None):
    if sub: fig.text(0.005, 0.965, sub, fontsize=8, color="#6B7280", va="top")
    fig.suptitle(title, fontsize=11, fontweight="bold", x=0.005, ha="left", y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.93 if sub else 0.96])
    path = OUT_DIR / f"{name}.png"
    fig.savefig(path, format="png", bbox_inches="tight", dpi=130)
    plt.close(fig)
    SAVED.append(dict(chart=name, title=title, path=str(path)))
def barh_err(labels, vals, lo, hi, colors, name, title, sub=None, xlab="", fmt=f4):
    if not HAVE_MPL: return
    n = len(labels)
    fig, ax = plt.subplots(figsize=(9.5, max(3.0, 0.28*n + 1.2)))
    y = np.arange(n)[::-1]
    v = np.asarray(vals, float)
    ax.barh(y, v, color=colors, height=0.72)
    if lo is not None:
        ax.errorbar(v, y, xerr=[v - np.asarray(lo, float), np.asarray(hi, float) - v],
                    fmt="none", ecolor=INK, elinewidth=.8, capsize=2)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7.5)
    ax.axvline(0, color=INK, lw=.8); ax.set_xlabel(xlab)
    for yi, vi in zip(y, v):
        if np.isfinite(vi): ax.text(vi, yi, " " + fmt(vi), va="center", fontsize=6.8)
    _save(fig, name, title, sub)
def bars(df, xcol, ycols, name, title, sub=None, fmt=usd, colors=None, stacked=False,
         ylab="", figsize=(9.5, 4.2)):
    if not HAVE_MPL: return
    fig, ax = plt.subplots(figsize=figsize)
    idx = np.arange(len(df)); n = len(ycols); w = 0.8 if stacked else 0.8/n
    bottom = np.zeros(len(df))
    for i, c in enumerate(ycols):
        v = df[c].to_numpy(float)
        x = idx if stacked else idx + (i-(n-1)/2)*w
        ax.bar(x, v, w, bottom=bottom if stacked else None, label=str(c),
               color=(colors or {}).get(c))
        if not stacked:
            for xi, vi in zip(x, v):
                if np.isfinite(vi): ax.text(xi, vi, fmt(vi), ha="center", va="bottom", fontsize=6.8)
        else: bottom = bottom + np.nan_to_num(v)
    ax.set_xticks(idx); ax.set_xticklabels([str(s) for s in df[xcol]], fontsize=8)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: fmt(v))); ax.set_ylabel(ylab)
    ax.margins(y=.15); ax.legend(frameon=False, fontsize=7.5, ncol=min(n, 4))
    _save(fig, name, title, sub)

# ── model (verbatim from 10_CORE_CODE §3–§4, plus warm start) ─────────
def auc(y, s):
    y = np.asarray(y, float); s = np.asarray(s, float)
    ok = np.isfinite(s) & np.isfinite(y); y, s = y[ok], s[ok]
    n1 = float(y.sum()); n0 = float(len(y) - n1)
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1*(n1+1)/2.0)/(n1*n0))

def logit_newton(X, y, sw, l2=L2, mi=100, tol=1e-8, init=None):
    """Weighted ridge logistic regression by DAMPED Newton (v10c). Each step is
    halved until the penalised log-likelihood improves; convergence is judged
    by the gradient. v11 adds `init` — a warm start for the drop-one refits.
    The objective is strictly concave, so the optimum does not depend on the
    start; §3a asserts that a warm and a cold fit agree."""
    X = np.asarray(X, np.float64); y = np.asarray(y, np.float64); sw = np.asarray(sw, np.float64)
    k = X.shape[1]; R = l2*np.eye(k); R[0, 0] = 1e-8       # intercept effectively unpenalised
    if init is not None:
        b = np.asarray(init, np.float64).copy()
    else:
        b = np.zeros(k)
        pbar = float(np.clip(np.average(y, weights=sw), 1e-6, 1-1e-6))
        b[0] = math.log(pbar/(1-pbar))
    def obj(bb):
        eta = X @ bb
        return float(np.sum(sw*(y*eta - np.logaddexp(0.0, eta))) - 0.5*bb @ R @ bb)
    f, it = obj(b), 0
    for it in range(1, mi+1):
        eta = X @ b; mu = 0.5*(1.0 + np.tanh(0.5*eta))
        g = X.T @ (sw*(y-mu)) - R @ b
        Hs = (X.T*(sw*mu*(1-mu))) @ X + R
        try: step = np.linalg.solve(Hs, g)
        except np.linalg.LinAlgError: step = np.linalg.lstsq(Hs, g, rcond=None)[0]
        t = 1.0
        while True:
            bn = b + t*step; fn = obj(bn)
            if fn >= f or t < 1e-10: break
            t *= 0.5
        if fn < f: break
        small = np.max(np.abs(bn-b)) < tol
        b, f = bn, fn
        if small: break
    mu = 0.5*(1.0 + np.tanh(0.5*(X @ b)))
    gmax = float(np.max(np.abs(X.T @ (sw*(y-mu)) - R @ b))/len(y))
    return b, dict(converged=gmax < 1e-6, iters=it, max_grad=gmax)

def fit_spec(tr, cols, l2=L2, wcol="sw", warm=None):
    """Inverse-probability-weighted fit (v10c). No King & Zeng offset. v11 also
    returns the standardisation (mu, sd) and the standardised coefficients, which
    the reason codes and the warm starts need. `warm` is an earlier spec fitted
    on the SAME rows: standardisation is per column, so its standardised
    coefficients are a valid start for any subset of its columns."""
    X = tr[cols].to_numpy(np.float64); y = tr["y"].to_numpy(np.float64)
    sw = (tr[wcol].to_numpy(np.float64) if wcol in tr.columns else np.ones(len(tr)))
    sw = sw/sw.mean()
    keep = X.std(axis=0) > 1e-9
    ck = [c for c, k in zip(cols, keep) if k]
    if not ck or y.sum() < 2: return None
    Xk = X[:, keep]
    mu = np.average(Xk, axis=0, weights=sw)
    sd = np.sqrt(np.average((Xk-mu)**2, axis=0, weights=sw)); sd[sd < 1e-9] = 1.0
    init = None
    if warm is not None:
        wm = dict(zip(warm["cols"], warm["b_std"]))
        init = np.concatenate([[warm["b0_std"]], [wm.get(c, 0.0) for c in ck]])
    b, info = logit_newton(np.column_stack([np.ones(len(Xk)), (Xk-mu)/sd]), y, sw, l2, init=init)
    return dict(cols=ck, beta=b[1:]/sd, b0=float(b[0] - np.sum(b[1:]*mu/sd)),
                mu=mu, sd=sd, b_std=b[1:], b0_std=float(b[0]), **info)

def predict_p(sp, df):
    if sp is None: return np.full(len(df), np.nan)
    eta = sp["b0"] + df[sp["cols"]].to_numpy(np.float64) @ sp["beta"]
    return 1.0/(1.0+np.exp(-np.clip(eta, -30, 30)))

def pav(x, y, w=None, nbins=200):
    """Weighted isotonic regression on quantile bins (v10c)."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    w = np.ones_like(y) if w is None else np.asarray(w, float)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0)
    x, y, w = x[ok], y[ok], w[ok]
    if len(x) == 0: return np.array([0.0]), np.array([0.0])
    nb = int(min(nbins, max(2, len(x)//50)))
    edges = np.unique(np.quantile(x, np.linspace(0, 1, nb+1)))
    if len(edges) < 3:
        return (np.array([float(x.min()), float(x.max())]),
                np.array([float(np.average(y, weights=w))]*2))
    idx = np.clip(np.searchsorted(edges, x, side="right")-1, 0, len(edges)-2)
    g = (pd.DataFrame({"b": idx, "yw": y*w, "xw": x*w, "w": w}).groupby("b")
         .agg(yw=("yw", "sum"), xw=("xw", "sum"), w=("w", "sum")))
    g["x"] = g.xw/g.w; g["y"] = g.yw/g.w
    g = g.sort_values("x")
    sx, sy, sw_ = [], [], []
    for xi, yi, wi in zip(g.x.to_numpy(), g.y.to_numpy(), g.w.to_numpy()):
        sx.append(float(xi)); sy.append(float(yi)); sw_.append(float(wi))
        while len(sy) > 1 and sy[-2] > sy[-1]:
            nw = sw_[-2] + sw_[-1]
            sy[-2] = (sy[-2]*sw_[-2] + sy[-1]*sw_[-1])/nw
            sw_[-2] = nw; sx[-2] = sx[-1]
            sy.pop(); sw_.pop(); sx.pop()
    return np.asarray(sx), np.asarray(sy)

def apply_iso(knots, p):
    kx, ky = knots; p = np.asarray(p, float)
    if len(ky) == 0 or not np.isfinite(ky).any() or np.nanmax(ky) <= 0: return p
    return np.interp(p, kx, ky, left=ky[0], right=ky[-1])

def label(d, defn, H):
    """Forward window only; unobservable rows dropped (v10c)."""
    t = pd.to_numeric(d["m_idx"], errors="coerce")
    if defn == "AB_any":
        ev = pd.concat([pd.to_numeric(d["event_A"], errors="coerce"),
                        pd.to_numeric(d["event_B"], errors="coerce")], axis=1).min(axis=1)
    elif defn == "A_full_exit":
        ev = pd.to_numeric(d["event_A"], errors="coerce")
    else:
        raise ValueError(defn)
    y = ((ev > t) & (ev <= t + H)).astype(float)
    keep = ((t + H <= M_MAX) | (y == 1)) & (ev.isna() | (ev > t))
    o = d.loc[keep].copy()
    o["y"] = y.loc[keep].to_numpy(); o["event_m"] = ev.loc[keep].to_numpy()
    return o

# ── the queue and the paired comparison (v10d §5, generalised to any p_ column) ──
def qscore(p, bal, alpha=ALPHA):
    return np.asarray(p, float)*np.power(np.maximum(np.nan_to_num(np.asarray(bal, float)), 1.0), alpha)
def first_alerts(df, score_col, K, alpha=0.0, rank_col="bar_med12"):
    s = pd.to_numeric(df[score_col], errors="coerce").to_numpy(float)
    if alpha > 0:
        b = np.maximum(pd.to_numeric(df[rank_col], errors="coerce").fillna(0.0).to_numpy(float), 1.0)
        s = s*np.power(b, alpha)
    d = df.assign(_s=s)
    d["_r"] = d.groupby("m_idx")["_s"].rank(ascending=False, method="first", na_option="bottom")
    fl = d[d._r <= K]
    first = fl.sort_values(["cust_pwr_id", "m_idx"]).drop_duplicates("cust_pwr_id", keep="first")
    return fl, first
def queue_tp(frame, name, alpha=ALPHA, K=QUEUE_K, rank_col="bar_med12"):
    _, first = first_alerts(frame, f"p_{name}", K, alpha, rank_col)
    return first, first[first.y == 1]
def topk_hits(te, p, alpha=ALPHA, K=QUEUE_K):
    """Positive client-months inside the top K of ONE month — the queue-level
    counterpart of AUC, which averages the whole ranking."""
    s = qscore(p, te["bar_med12"].to_numpy(), alpha)
    k = min(K, len(s))
    if k == 0: return 0
    top = np.argpartition(-np.nan_to_num(s, nan=-np.inf), k-1)[:k]
    return int(te["y"].to_numpy()[top].sum())
try:
    from scipy import stats as _st
    def tcrit(df): return float(_st.t.ppf(0.975, df)) if df > 0 else np.nan
except Exception:
    _TT = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306}
    def tcrit(df): return _TT.get(df, 1.96)
def contrib(frame, name, alpha, credit="bar_cap"):
    _, tp = queue_tp(frame, name, alpha)
    return tp.groupby("cust_pwr_id")[credit].sum().astype(float)
def paired(frame, fx, fy, alpha=ALPHA, credit="bar_cap", seed=SEED):
    """v10d §5: client bootstrap + whale jackknife + month by month, on any
    two score columns p_{fx} and p_{fy} of the same frame."""
    cx, cy = contrib(frame, fx, alpha, credit), contrib(frame, fy, alpha, credit)
    ix = cx.index.union(cy.index)
    d = (cx.reindex(ix, fill_value=0.0) - cy.reindex(ix, fill_value=0.0)).to_numpy()
    D, n = float(d.sum()), len(d)
    rng = np.random.default_rng(seed)
    Db = np.zeros(N_BOOT)
    if n > 0:
        for s in range(0, N_BOOT, 500):
            m = min(500, N_BOOT - s)
            Db[s:s+m] = d[rng.integers(0, n, size=(m, n))].sum(axis=1)
    lo, hi = np.percentile(Db, [2.5, 97.5])
    lead = 1.0 if D >= 0 else -1.0
    rem = abs(D) - np.cumsum(np.sort(d*lead)[::-1]) if n else np.array([])
    flip = int(np.argmax(rem <= 0) + 1) if (rem <= 0).any() else None
    md = []
    for T, g in frame.groupby("m_idx"):
        v = []
        for nm in (fx, fy):
            s_ = qscore(pd.to_numeric(g[f"p_{nm}"], errors="coerce"), g.bar_med12, alpha)
            top = g.assign(_s=s_).nlargest(QUEUE_K, "_s")
            v.append(float(top.loc[top.y == 1, credit].sum()))
        md.append(v[0] - v[1])
    md = np.asarray(md); k = len(md)
    se = md.std(ddof=1)/np.sqrt(k) if k > 1 else np.nan
    return dict(comparison=f"{fx} − {fy}", clients_in_play=n, difference=D,
                ci_lo=float(lo), ci_hi=float(hi), p_first_ahead=float((Db > 0).mean()),
                clients_to_flip=("never" if flip is None else flip),
                months_first_wins=f"{int((md > 0).sum())} of {k}",
                month_ci_lo=float(md.mean() - tcrit(k-1)*se) if k > 1 else np.nan,
                month_ci_hi=float(md.mean() + tcrit(k-1)*se) if k > 1 else np.nan)

# ── signals: an ld_ value and its md_ missingness flag are ONE signal ──
_BASE_DESC = {
    "bal_live": "balance on live accounts", "amt_out": "$ paid out", "amt_in": "$ received",
    "net_flow": "net flow (in − out)", "n_out": "number of payments out",
    "n_in": "number of payments in", "avg_ticket_out": "average payment size out",
    "avg_ticket_in": "average receipt size in", "amt_out_ach": "$ out by ACH",
    "amt_out_wire": "$ out by wire", "amt_out_check": "$ out by cheque",
    "amt_out_card": "$ out by card", "amt_out_rtp": "$ out by RTP", "amt_out_other": "$ out, other rails",
    "amt_out_internal": "$ paid to other PNC customers", "amt_in_internal": "$ received from PNC customers",
    "cpty_out_n": "counterparties paid", "fin_out_n": "institutions paid by ACH",
    "cpty_new_out": "new counterparties paid", "fin_new_out": "new institutions paid",
    "bl_log_bal": "balance level (log)", "bl_bal_cv12": "balance volatility, 12m",
    "bl_bal_vs_peak": "balance vs 12m peak", "bl_bal_share_top_acct": "share held in largest account",
    "acc_live_ratio": "share of accounts still live", "acc_closed_share": "share of accounts closed"}
_PFX_DESC = [("fin_in", "institutions paying in"), ("cptya_out", "counterparties paid (account key)"),
             ("fin_out2", "institutions paid out"), ("cptyn_out", "counterparties paid (name key)"),
             ("cptyn_in", "payers (name key)"), ("cptya_in", "payers (account key)"),
             ("cpty_in", "payers")]
_SFX_DESC = [("_rec_broken_amt_share", "share of standing $ that stopped"),
             ("_rec_broken_share", "share of standing relationships that stopped"),
             ("_rec_broken_amt", "standing $ that stopped"), ("_rec_broken", "standing relationships stopped"),
             ("_rec_active_amt", "$ in standing relationships"), ("_rec_active", "standing relationships active"),
             ("_retention", "retained vs recent months"), ("_lost_k", "lost vs recent months"),
             ("_new_k", "new vs recent months"), ("_n_k", "count")]
def sig_of(col): return col[3:]
def family_of(sig):
    if sig == "bal_live" or sig.startswith(("bl_", "acc_")): return "deposit"
    for p in ("fin_in", "cptya_out", "fin_out2"):
        if sig.startswith(p): return p
    if "rec_" in sig: return "recurring"
    return "payment_v7"
def describe(sig):
    if sig in _BASE_DESC: return _BASE_DESC[sig]
    head = next((d for p, d in _PFX_DESC if sig.startswith(p)), None)
    tail = next((d for s, d in _SFX_DESC if sig.endswith(s)), None)
    if head and tail: return f"{head}: {tail}"
    if tail: return f"{sig.split('_rec_')[0]}: {tail}"
    return head or sig
print(f"helpers ready · charts {'saved to ' + str(OUT_DIR) if HAVE_MPL else 'OFF'}")

In [ ]:
# =====================================================================
# 1 · DATA — risk set, all_features only (Spark)          [OUTPUT BLOCK 1]
# =====================================================================
# Same construction and the SAME deterministic case-control sample as v10c §3,
# so the refit in §2 can be checked against v10c's numbers.
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
if "spark" not in globals():
    spark = (SparkSession.builder.appName("pkg_attrition_eda_v11")
             .config("spark.sql.shuffle.partitions", "800")
             .config("spark.sql.execution.arrow.pyspark.enabled", "false")
             # KEEP OFF: PySpark 3.3's arrow path uses np.object0 / np.bool8 (numpy 2)
             .enableHiveSupport().getOrCreate())
from pyspark.sql import types as _ST   # NOT `T`: T is the origin loop variable everywhere
def to_sdf(pdf):
    """pandas -> Spark WITHOUT spark.createDataFrame(pandas). On the cluster
    (PySpark 3.3, pandas 2.x) that path calls DataFrame.iteritems, which pandas
    2.0 removed — and the session has Arrow on, whose fallback hits the same
    call. Rows go over as plain Python tuples with an explicit schema."""
    fields, cols = [], []
    for c in pdf.columns:
        s, k = pdf[c], pdf[c].dtype.kind
        if k in "iub":
            fields.append(_ST.StructField(c, _ST.LongType())); cols.append(s.astype("int64").tolist())
        elif k == "f":
            fields.append(_ST.StructField(c, _ST.DoubleType())); cols.append(s.astype(float).tolist())
        else:
            fields.append(_ST.StructField(c, _ST.StringType()))
            cols.append([None if (v is None or (isinstance(v, float) and v != v)) else str(v) for v in s.tolist()])
    return spark.createDataFrame(list(zip(*cols)) if cols else [], _ST.StructType(fields))
def collect_pd(sdf, lbl=""):
    t0 = time.time(); out = _dec(sdf).toPandas()
    print(f"  collected {lbl}: {len(out):,} x {out.shape[1]} in {time.time()-t0:,.0f}s")
    return out
def hdfs_exists(p):
    jvm = spark._jvm; conf = spark._jsc.hadoopConfiguration()
    path = jvm.org.apache.hadoop.fs.Path(p)
    return bool(path.getFileSystem(conf).exists(path))

t_data = time.time()
cust_month = spark.read.parquet(hp("v2", "panel_customer_month"))
M_MIN, M_MAX = [int(x) for x in cust_month.agg(F.min("m_idx"), F.max("m_idx")).collect()[0]]
# m_idx = year*12 + month. §5 builds months from the payment table on that
# convention, so it is asserted here rather than assumed.
_y0, _m0 = int(DATE_START[:4]), int(DATE_START[5:7])
assert M_MIN == _y0*12 + _m0, (f"m_idx convention broken: M_MIN={M_MIN}, expected "
                               f"{_y0*12 + _m0} for {DATE_START}")
ORIGINS = list(range(M_MIN + ORIGIN_START_OFF, M_MAX - PRIMARY_H + 1))
assert 1 <= len(ORIGINS) <= 24, (f"{len(ORIGINS)} origins — m_idx is ABSOLUTE "
                                 f"({M_MIN}-{M_MAX}); ORIGIN_START_OFF is an OFFSET")
ANNUALISE = 12.0/len(ORIGINS)

lab  = spark.read.parquet(hp("v6", "labels_customer")).persist(StorageLevel.DISK_ONLY)
BAL  = spark.read.parquet(hp("v9", "balance")).persist(StorageLevel.DISK_ONLY)
RISK = spark.read.parquet(hp("v9", "risk_set_v9"))
ALLC = RISK.columns
_missing = {"cust_pwr_id", "m_idx", "event_A", "event_B", "bar_now", "bar_med12"} - set(ALLC)
assert not _missing, f"risk_set_v9 lacks {_missing}"

# all_features — identical construction to v10 / v10b / v10c
NEWP = ("cptyn_", "cptya_", "fin_out2", "fin_in", "tim_", "railmix", "conc_",
        "selfpay", "acc_", "bl_")
def _pair(ld): return sorted(set(ld + [f"md_{c[3:]}" for c in ld if f"md_{c[3:]}" in ALLC]))
def _blk(*pfx): return _pair(sorted([c for c in ALLC if any(c.startswith("ld_"+p) for p in pfx)]))
V7_LD = sorted([c for c in ALLC if c.startswith("ld_")
                and not any(c.startswith("ld_"+p) for p in NEWP)])
DEPOSIT = _pair([c for c in V7_LD if c == "ld_bal_live"]) + _blk("bl_") + _blk("acc_")
PAYMENT = _pair([c for c in V7_LD if c != "ld_bal_live"])
BOTH    = sorted(set(DEPOSIT + PAYMENT))
COLS    = sorted(set(BOTH + _blk("fin_in") + _blk("cptya_out") + _blk("fin_out2") +
                     [c for c in ALLC if "rec_" in c and c.startswith(("ld_", "md_"))]))
assert not (set(DEPOSIT) & set(PAYMENT)), "deposit and payment sets must be disjoint"
if len(COLS) != 148:
    print(f"  ⚠ all_features has {len(COLS)} columns, v10c had 148 — the risk set changed")

NEED = sorted(set(["cust_pwr_id", "m_idx", "event_A", "event_B", "bar_now", "bar_med12"] + COLS)
              & set(ALLC))
RS = RISK.select(*NEED)
def _ahead(c, h):
    return F.coalesce(F.col(c).between(F.col("m_idx") + 1, F.col("m_idx") + h), F.lit(False))
# THE v10c RULE, unchanged: kept whole = A or B within H; else kept w.p. NEG_SAMPLE.
KEEP_WHOLE = _ahead("event_A", PRIMARY_H) | _ahead("event_B", PRIMARY_H)
TRS = (RS.filter(F.col("m_idx") <= max(ORIGINS) - PRIMARY_H)
       .withColumn("kept_whole", KEEP_WHOLE.cast("int"))
       .withColumn("_u", (F.abs(F.hash(F.concat_ws("|", "cust_pwr_id",
                   F.col("m_idx").cast("string"), F.lit(SEED)))) % 100000)/100000.0)
       .filter((F.col("kept_whole") == 1) | (F.col("_u") < NEG_SAMPLE))
       .withColumn("sw", F.when(F.col("kept_whole") == 1, F.lit(1.0))
                          .otherwise(F.lit(1.0/NEG_SAMPLE)))
       .drop("_u"))
TR_RAW = collect_pd(TRS, "TRAIN")
TE_RAW = {t: collect_pd(RS.filter(F.col("m_idx") == t), f"TEST {t}") for t in ORIGINS}

def prep(d):
    d = d.copy()
    d["cust_pwr_id"] = d["cust_pwr_id"].astype(str)          # ids are strings end to end
    d["m_idx"] = pd.to_numeric(d["m_idx"], errors="coerce").astype(int)
    for c in d.columns:
        if c.startswith("ld_"):   d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)
        elif c.startswith("md_"): d[c] = pd.to_numeric(d[c], errors="coerce").fillna(1.0)
    for c in ["bar_now", "bar_med12"]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0).clip(lower=0)
    for c, dv in [("sw", 1.0), ("kept_whole", 1)]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(dv) if c in d.columns else dv
    d["defend"] = (d.bar_now >= DEFEND_FRAC*d.bar_med12) & (d.bar_now >= DEFEND_MIN)
    return d
TRw = prep(TR_RAW); TE = {k: prep(v) for k, v in TE_RAW.items()}
del TR_RAW, TE_RAW

LABS = _dec(lab.select("cust_pwr_id", "q_A_full_exit", "q_B_bal_exit")).toPandas()
LABS = LABS.rename(columns={"q_A_full_exit": "qA", "q_B_bal_exit": "qB"})
LABS["cust_pwr_id"] = LABS.cust_pwr_id.astype(str)
LABS["qA"] = pd.to_numeric(LABS.qA, errors="coerce")
LABS["qB"] = pd.to_numeric(LABS.qB, errors="coerce")
assert LABS.cust_pwr_id.is_unique, "labels_customer must be one row per client"

SIGS = sorted({sig_of(c) for c in COLS})
PAIR = {s: [c for c in (f"ld_{s}", f"md_{s}") if c in COLS] for s in SIGS}
FAM = {s: family_of(s) for s in SIGS}
FAMS = [f for f in ["deposit", "payment_v7", "fin_in", "cptya_out", "fin_out2", "recurring"]
        if any(v == f for v in FAM.values())]
assert sum(len(v) for v in PAIR.values()) == len(COLS), "every column must belong to one signal"
assert set(COLS) >= set(DEPOSIT), "deposit block must be inside all_features"
kv([("m_idx range", f"{M_MIN}–{M_MAX} · origins {ORIGINS[0]}–{ORIGINS[-1]} "
                    f"({len(ORIGINS)} test months) · annualise ×{ANNUALISE:.3f}"),
    ("all_features columns", f"{len(COLS)} = {len(SIGS)} signals (value + missingness flag)"),
    ("signals by family", " · ".join(f"{f} {sum(1 for v in FAM.values() if v == f)}" for f in FAMS)),
    ("training rows (case-control, IPW)", f"{len(TRw):,} · kept whole {int(TRw.kept_whole.sum()):,}"),
    ("test client-months (full book)", f"{sum(len(v) for v in TE.values()):,}"),
    ("…where money is still here", f"{sum(int(v.defend.sum()) for v in TE.values()):,}"),
    ("labelled clients", f"{len(LABS):,} · A {int(LABS.qA.notna().sum()):,} · "
                         f"B {int(LABS.qB.notna().sum()):,} · B never A "
                         f"{int((LABS.qB.notna() & LABS.qA.isna()).sum()):,}"),
    ("wall", f"{time.time()-t_data:,.0f}s")],
   title="1 &middot; <b>Data</b>", save="v11_1_data")

## 2 · The all-features model, refitted — and checked against v10c

Seven rolling origins, money still here in training **and** test, IPW + damped Newton, weighted isotonic on
the last two training months. Every later section needs the fitted coefficients, which v10c did not
persist, so the model is refitted rather than loaded. The refit uses the same deterministic sample, so it
must land on v10c's AUC (0.7483) and v10d's capped headline (229 departing clients, $606.8m) — if it does
not, the data changed and nothing below should be compared with earlier versions.

In [ ]:
# =====================================================================
# 2 · THE ALL-FEATURES MODEL — refit + reproduction check  [OUTPUT BLOCK 2]
# =====================================================================
def fold_frames(T):
    """(fit, calibration, test) frames for origin T, exactly as v10c §4 built
    them for (A_full_exit, defend). Deterministic, so every section rebuilds
    them rather than holding seven copies of the training frame in memory."""
    trf = label(TRw[TRw.m_idx <= T - PRIMARY_H], DEFN, PRIMARY_H)
    trf = trf[trf.defend]
    if len(trf) == 0 or trf.y.sum() < MIN_TRAIN_POS: return None
    cut = trf.m_idx.max() - CAL_MONTHS
    fit_df, cal_df = trf[trf.m_idx <= cut], trf[trf.m_idx > cut]
    if len(cal_df) < 3000 or cal_df.y.sum() < 20 or fit_df.y.sum() < MIN_TRAIN_POS:
        fit_df = cal_df = trf                        # degrade rather than fail (v10c)
    te = label(TE[T], DEFN, PRIMARY_H); te = te[te.defend]
    if len(te) == 0 or te.y.sum() < 1: return None
    return fit_df, cal_df, te

t0 = time.time()
SPEC, KNOTS, FOLDROWS, _sc = {}, {}, [], []
for T in ORIGINS:
    fr = fold_frames(T)
    if fr is None:
        print(f"  origin {T}: skipped — too few positives"); continue
    fit_df, cal_df, te = fr
    sp = fit_spec(fit_df, COLS)
    assert sp is not None, f"origin {T}: fit returned nothing"
    knots = pav(predict_p(sp, cal_df), cal_df.y.to_numpy(), cal_df.sw.to_numpy())
    praw = predict_p(sp, te); pc = apply_iso(knots, praw)
    SPEC[T], KNOTS[T] = sp, knots
    yv = te.y.to_numpy()
    FOLDROWS.append(dict(origin=T, n_fit=len(fit_df), fit_pos=int(fit_df.y.sum()),
                         n_cal=len(cal_df), test_rows=len(te), test_pos=int(yv.sum()),
                         n_feat=len(sp["cols"]), auc=auc(yv, praw),
                         hits_in_top_K=topk_hits(te, pc), converged=bool(sp["converged"]),
                         iters=sp["iters"], max_grad=sp["max_grad"]))
    o = te[["cust_pwr_id", "m_idx", "y", "event_m", "bar_now", "bar_med12"]].copy()
    o["praw_full"], o["p_full"] = praw, pc
    _sc.append(o)
FOLDS = pd.DataFrame(FOLDROWS)
ORIG = FOLDS.origin.tolist()
assert len(ORIG) >= 2, "fewer than two usable origins"
_bad = FOLDS[~FOLDS.converged]
assert _bad.empty, ("non-converged fits — do not read anything downstream:\n"
                    + _bad[["origin", "iters", "max_grad"]].to_string())
SCF = pd.concat(_sc, ignore_index=True); del _sc
SCF["bar_cap"] = np.minimum(SCF.bar_now, SCF.bar_med12)     # the conservative credit (v10d)
save_frame(SCF, "v11_scored_all_features_defend")

# calibration: realised / predicted in the top predicted decile (test rows are the
# full money-still-here book, unsampled, so no weights)
_d = SCF[["y", "p_full"]].dropna()
_d["dec"] = pd.qcut(_d.p_full.rank(method="first"), 10, labels=False) + 1
_g = _d.groupby("dec").agg(pred=("p_full", "mean"), real=("y", "mean"))
cal_top = float(_g.real.iloc[-1]/_g.pred.iloc[-1]) if _g.pred.iloc[-1] > 0 else np.nan

Q_FIRST, Q_TP = queue_tp(SCF, "full")
reach, conv = float(Q_TP.bar_cap.sum()), int(len(Q_FIRST))
RES["auc_refit"] = float(FOLDS.auc.mean())
RES["repro_dollars_ratio"] = reach/REF_DOLLARS

disp(FOLDS.assign(auc=FOLDS.auc.round(4), max_grad=FOLDS.max_grad.map(lambda v: f"{v:.1e}")),
     title="2a &middot; <b>Seven folds, money still here</b> — every fit converged", n=12,
     save="v11_2a_folds")
kv([("mean AUC (money still here)", f"{RES['auc_refit']:.4f}  ·  v10c {REF_AUC_DEFEND:.4f}  ·  "
                                    f"Δ {RES['auc_refit'] - REF_AUC_DEFEND:+.4f}"),
    ("AUC range across folds", f"{FOLDS.auc.min():.4f} – {FOLDS.auc.max():.4f}"),
    ("top-decile calibration, realised ÷ predicted", f"{cal_top:.3f}  (v10c 0.99–1.06)"),
    ("queue: conversations (K=1,000, α=0.5)", f"{conv:,}"),
    ("queue: departing clients reached", f"{len(Q_TP):,}  ·  v10d {REF_TP_CLIENTS}"),
    ("queue: capped dollars reached", f"{usd(reach)}  ·  v10d {usd(REF_DOLLARS)}  ·  "
                                      f"ratio {RES['repro_dollars_ratio']:.3f}"),
    ("retained at a 1% save rate, per year", usd(reach*0.01*ANNUALISE)),
    ("break-even save rate", pctf(conv*RM_COST_PER_CALL/max(reach, 1.0), 3)),
    ("fit wall", f"{time.time()-t0:,.0f}s")],
   title="2b &middot; <b>Does the refit reproduce v10c / v10d?</b>", save="v11_2b_reproduction")
if abs(RES["auc_refit"] - REF_AUC_DEFEND) > 0.003 or abs(RES["repro_dollars_ratio"] - 1) > 0.05:
    print("  ⚠ the refit does not reproduce v10c/v10d — the inputs changed since 22 Sep 2026. "
          "Rankings below are internally consistent but not comparable with earlier numbers.")

## 3 · Ranking the signals by predictive power

Each of the 74 signals is one unit — its `ld_` value and its `md_` missingness flag enter and leave the
model together, because splitting them lets the flag carry the value's information and the ranking then
credits the wrong column.

| Lens | Question | How | What would fool it |
|---|---|---|---|
| **Standalone** AUC | Does it predict on its own? | a two-column model per fold, same weights and risk set | correlation with a stronger signal — it inherits their lift |
| **Unique** ΔAUC (drop-one) | Does the model *need* it? | refit without it on the same rows, warm-started from the full fit, score the same test month | a near-copy elsewhere in the set absorbs it — both look useless |
| **Reliance** ΔAUC (permutation) | How hard does *this* fitted model lean on it? | shuffle its rows in the test month; coefficients untouched | a signal the model uses but that could be replaced — reliance ≠ necessity |

All three are measured on the seven test months (never the training rows) and summarised by mean,
per-fold spread and the number of folds in which the effect has the expected sign. The **consensus rank**
is the mean of the three ranks. Tiers:

- **core** — the model loses AUC without it (mean ΔAUC ≥ 0.0005 and positive in ≥ 6 of 7 folds)
- **harmful** — the model is *better* without it (mean ΔAUC ≤ −0.0005)
- **redundant** — predicts alone (|AUC − 0.5| ≥ 0.05) but adds nothing the others don't already carry
- **weak** — neither

A coefficient's sign is reported but never used to rank: in a ridge model with correlated inputs a sign
can flip against the signal's standalone direction, which is flagged rather than explained away.

In [ ]:
# =====================================================================
# 3a · THREE LENSES PER SIGNAL — standalone, drop-one, permutation [OUTPUT BLOCK 3]
# =====================================================================
DROP_CACHE = OUT_DIR / "v11_3a_drop_one_cache.csv"
_cache = pd.read_csv(DROP_CACHE) if (RUN_DROP_ONE and DROP_CACHE.exists()) else pd.DataFrame()
if len(_cache):
    _cache = _cache[_cache.sig.isin(SIGS) & _cache.origin.isin(ORIG)]
_done = set(zip(_cache.origin.astype(int), _cache.sig)) if len(_cache) else set()
DROP_ROWS = _cache.to_dict("records") if len(_cache) else []

def _perm_delta(sp, Xte, yv, eta, sig, rng, reps=PERM_REPS):
    """Reliance: shuffle the signal's rows (value and flag together) in the test
    month and re-score by adjusting eta — no refit, no frame copies."""
    idx = [i for i, c in enumerate(sp["cols"]) if c in PAIR[sig]]
    if not idx: return 0.0
    Xs = Xte[:, idx]; bs = sp["beta"][idx]
    base = eta - Xs @ bs
    a0 = auc(yv, eta)
    return float(a0 - np.mean([auc(yv, base + Xs[rng.permutation(len(yv))] @ bs) for _ in range(reps)]))

LENS_ROWS, SELF_CHECK = [], {}
rng = np.random.default_rng(SEED)
t0, n_fit = time.time(), 0
for T in ORIG:
    fit_df, cal_df, te = fold_frames(T)
    sp = SPEC[T]; yv = te.y.to_numpy()
    Xte = te[sp["cols"]].to_numpy(np.float64)
    eta = sp["b0"] + Xte @ sp["beta"]
    base_auc = auc(yv, eta)
    base_hits = int(FOLDS.loc[FOLDS.origin == T, "hits_in_top_K"].iloc[0])
    bstd = dict(zip(sp["cols"], sp["b_std"]))
    # warm vs cold self-check, once: the objective is strictly concave, so a warm
    # start may only change the iteration count — never the answer
    if not SELF_CHECK:
        s0 = SIGS[0]
        _cols = [c for c in COLS if c not in PAIR[s0]]
        cold = fit_spec(fit_df, _cols); warm = fit_spec(fit_df, _cols, warm=sp)
        SELF_CHECK = dict(sig=s0, iters_cold=cold["iters"], iters_warm=warm["iters"],
                          max_coef_diff=float(np.max(np.abs(cold["b_std"] - warm["b_std"]))),
                          auc_diff=abs(auc(yv, predict_p(cold, te)) - auc(yv, predict_p(warm, te))))
        assert SELF_CHECK["max_coef_diff"] < 1e-5 and SELF_CHECK["auc_diff"] < 1e-6, \
            f"warm start changed the fit: {SELF_CHECK}"
    for s in SIGS:
        pc = PAIR[s]
        # 1 · standalone
        sa = fit_spec(fit_df, pc)
        a_alone = auc(yv, predict_p(sa, te)) if sa is not None else np.nan
        ld = f"ld_{s}"
        dir_alone = (float(np.sign(dict(zip(sa["cols"], sa["b_std"])).get(ld, np.nan)))
                     if sa is not None else np.nan)
        # 2 · drop-one (cached)
        if RUN_DROP_ONE and (T, s) not in _done:
            spd = fit_spec(fit_df, [c for c in COLS if c not in pc], warm=sp); n_fit += 1
            praw_d = predict_p(spd, te)
            knots_d = pav(predict_p(spd, cal_df), cal_df.y.to_numpy(), cal_df.sw.to_numpy())
            DROP_ROWS.append(dict(origin=T, sig=s, auc_drop=auc(yv, praw_d),
                                  hits_drop=topk_hits(te, apply_iso(knots_d, praw_d)),
                                  converged=bool(spd["converged"]), iters=spd["iters"]))
            _done.add((T, s))
        # 3 · permutation
        d_perm = _perm_delta(sp, Xte, yv, eta, s, rng)
        md = f"md_{s}"
        cov = (float((te[md] == 0).mean()) if md in te.columns
               else float((te[ld] != 0).mean()) if ld in te.columns else np.nan)
        LENS_ROWS.append(dict(origin=T, sig=s, fam=FAM[s], base_auc=base_auc, base_hits=base_hits,
                              auc_alone=a_alone, dir_alone=dir_alone, d_perm=d_perm,
                              b_std=bstd.get(ld, np.nan), coverage=cov))
    if RUN_DROP_ONE:
        pd.DataFrame(DROP_ROWS).to_csv(DROP_CACHE, index=False)   # resumable after every fold
    print(f"  origin {T}: {len(SIGS)} signals · {n_fit} drop-one refits so far · {time.time()-t0:,.0f}s")

LENS = pd.DataFrame(LENS_ROWS)
if RUN_DROP_ONE:
    DROPS = pd.DataFrame(DROP_ROWS)
    _nc = DROPS[~DROPS.converged.astype(bool)]
    assert _nc.empty, f"{len(_nc)} drop-one refits did not converge:\n{_nc.head().to_string()}"
    LENS = LENS.merge(DROPS[["origin", "sig", "auc_drop", "hits_drop"]], on=["origin", "sig"], how="left")
    LENS["d_drop"] = LENS.base_auc - LENS.auc_drop
    LENS["d_hits"] = LENS.base_hits - LENS.hits_drop
else:
    LENS["d_drop"] = np.nan; LENS["d_hits"] = np.nan
LENS.to_csv(OUT_DIR / "v11_3a_lens_by_fold.csv", index=False)

# ── per-signal summary ────────────────────────────────────────────────
nF = LENS.origin.nunique()
core_folds = min(CORE_FOLDS, max(nF - 1, 1))
def _summ(g):
    b = g.b_std.dropna(); sgn = np.sign(b.mean()) if len(b) else np.nan
    dd = g.d_drop.dropna()
    return pd.Series(dict(
        family=g.fam.iloc[0], auc_alone=g.auc_alone.mean(),
        d_drop=dd.mean() if len(dd) else np.nan,
        d_drop_lo=dd.mean() - tcrit(len(dd)-1)*dd.std(ddof=1)/np.sqrt(len(dd)) if len(dd) > 1 else np.nan,
        d_drop_hi=dd.mean() + tcrit(len(dd)-1)*dd.std(ddof=1)/np.sqrt(len(dd)) if len(dd) > 1 else np.nan,
        drop_pos_folds=int((dd > 0).sum()), drop_neg_folds=int((dd < 0).sum()),
        d_hits=g.d_hits.mean(), d_perm=g.d_perm.mean(),
        b_std=b.mean() if len(b) else np.nan,
        sign_consistency=float((np.sign(b) == sgn).mean()) if len(b) else np.nan,
        dir_alone=np.sign(g.dir_alone.mean()) if g.dir_alone.notna().any() else np.nan,
        coverage=g.coverage.mean()))
RANK = (LENS.groupby("sig")[["fam", "auc_alone", "d_drop", "d_hits", "d_perm", "b_std", "dir_alone",
                             "coverage"]].apply(_summ).reset_index())
RANK["description"] = RANK.sig.map(describe)
RANK["r_alone"] = RANK.auc_alone.sub(0.5).abs().rank(ascending=False, method="average")
RANK["r_perm"] = RANK.d_perm.rank(ascending=False, method="average")
_lens = ["r_alone", "r_perm"]
if RUN_DROP_ONE:
    RANK["r_drop"] = RANK.d_drop.rank(ascending=False, method="average"); _lens.insert(1, "r_drop")
RANK["consensus"] = RANK[_lens].mean(axis=1)
RANK = RANK.sort_values(["consensus", "r_perm"]).reset_index(drop=True)
RANK.insert(0, "rank", np.arange(1, len(RANK) + 1))
predictive_alone = RANK.auc_alone.sub(0.5).abs() >= REDUNDANT_MIN
if RUN_DROP_ONE:
    core = (RANK.d_drop >= DROP_MIN) & (RANK.drop_pos_folds >= core_folds)
    harmful = RANK.d_drop <= -DROP_MIN
else:   # without refits, reliance stands in for necessity — flagged in the table title
    core = RANK.d_perm >= DROP_MIN
    harmful = pd.Series(False, index=RANK.index)
RANK["tier"] = np.select([core, harmful, predictive_alone], ["core", "harmful", "redundant"], "weak")
RANK["direction"] = np.where(RANK.b_std > 0, "higher → more risk",
                     np.where(RANK.b_std < 0, "higher → less risk", "—"))
RANK["sign_flip_vs_alone"] = (np.sign(RANK.b_std) != RANK.dir_alone) & RANK.dir_alone.notna() & (RANK.b_std != 0)
RANK.to_csv(OUT_DIR / "v11_3a_signal_ranking.csv", index=False)

if RUN_DROP_ONE:
    RES["share_no_unique"] = float((RANK.d_drop <= 0).mean())
    # pairwise-complete: one signal with no standalone AUC (constant in a fold) made
    # scipy return NaN for the whole correlation in the first cluster run
    _rr = RANK[["r_alone", "r_drop"]].dropna()
    RES["rank_spearman"] = float(_rr.corr(method="spearman").iloc[0, 1]) if len(_rr) > 2 else np.nan
kv([("signals × folds", f"{len(SIGS)} × {nF} · base AUC {LENS.groupby('origin').base_auc.first().mean():.4f}"),
    ("drop-one refits", (f"{len(LENS.dropna(subset=['d_drop'])):,} ({n_fit:,} this run, rest from cache) · "
                         f"all converged") if RUN_DROP_ONE else "OFF (RUN_DROP_ONE = False)"),
    ("warm vs cold self-check", f"{SELF_CHECK.get('sig')}: max coef diff {SELF_CHECK.get('max_coef_diff', np.nan):.1e}"
                                f" · iterations cold {SELF_CHECK.get('iters_cold')} → warm {SELF_CHECK.get('iters_warm')}"),
    ("permutation", f"{PERM_REPS} shuffles per signal per fold, value and flag together"),
    ("tiers", " · ".join(f"{t} {int((RANK.tier == t).sum())}" for t in ["core", "harmful", "redundant", "weak"])),
    ("signals whose removal costs nothing (mean ΔAUC ≤ 0)", pctf(RES.get("share_no_unique"))),
    ("Spearman, standalone rank vs unique rank", f"{RES.get('rank_spearman', np.nan):.3f}"),
    ("coefficient signs that flip vs standalone", f"{int(RANK.sign_flip_vs_alone.sum())} of {len(RANK)}"),
    ("wall", f"{time.time()-t0:,.0f}s")],
   title="3a &middot; <b>Ranking run</b>", save="v11_3a_run")

In [ ]:
# =====================================================================
# 3b · THE RANKING — tables inline, charts saved              [OUTPUT BLOCK 4]
# =====================================================================
def _fmt_rank(d):
    return d.assign(
        auc_alone=d.auc_alone.map(lambda v: f"{v:.3f}" if np.isfinite(v) else "—"),
        d_drop=[f"{f4(m)}  [{f4(lo)}, {f4(hi)}]  {int(p)}/{nF}+" if np.isfinite(m) else "—"
                for m, lo, hi, p in zip(d.d_drop, d.d_drop_lo, d.d_drop_hi, d.drop_pos_folds)],
        d_perm=d.d_perm.map(f4), d_hits=d.d_hits.map(lambda v: f"{v:+.1f}" if np.isfinite(v) else "—"),
        b_std=d.b_std.map(lambda v: f"{v:+.3f}" if np.isfinite(v) else "—"),
        sign_consistency=d.sign_consistency.map(lambda v: pctf(v, 0)),
        coverage=d.coverage.map(lambda v: pctf(v, 0)),
        sign_flip_vs_alone=d.sign_flip_vs_alone.map({True: "⚠ flips", False: ""}))
SHOWC = ["rank", "sig", "description", "family", "tier", "direction", "auc_alone", "d_drop",
         "d_perm", "d_hits", "b_std", "sign_consistency", "sign_flip_vs_alone", "coverage"]
disp(_fmt_rank(RANK)[SHOWC],
     title=f"3b &middot; <b>All {len(RANK)} signals, ranked by consensus of the three lenses.</b> "
           f"auc_alone = standalone AUC · d_drop = AUC lost when refitted without it "
           f"[95% t-interval over folds] and folds where removal hurt · d_perm = AUC lost when shuffled · "
           f"d_hits = top-{QUEUE_K:,} hits lost per month without it · b_std = standardised coefficient "
           f"in the full model" + ("" if RUN_DROP_ONE else " · <b>drop-one OFF: tiers use reliance</b>"),
     n=len(RANK), save="v11_3b_signal_ranking")

_fam = (RANK.groupby("family")
        .agg(signals=("sig", "size"), core=("tier", lambda t: int((t == "core").sum())),
             redundant=("tier", lambda t: int((t == "redundant").sum())),
             harmful=("tier", lambda t: int((t == "harmful").sum())),
             best_rank=("rank", "min"), median_rank=("rank", "median"),
             sum_d_drop=("d_drop", "sum"), sum_d_perm=("d_perm", "sum"))
        .reindex(FAMS).reset_index())
disp(_fam.assign(sum_d_drop=_fam.sum_d_drop.map(f4), sum_d_perm=_fam.sum_d_perm.map(f4)),
     title="3b &middot; <b>By family</b> — where the core signals come from. Summed single-signal "
           "ΔAUCs understate a family's worth when its members cover for each other; §3c drops "
           "families whole", save="v11_3b_by_family")

# charts — saved, never shown
if HAVE_MPL:
    top = RANK.head(25)
    lab_ = [f"{r.rank:>2}. {r.sig}  ({r.tier})" for r in top.itertuples()]
    col_ = [FAMCOL.get(f, GREY) for f in top.family]
    if RUN_DROP_ONE:
        barh_err(lab_, top.d_drop, top.d_drop_lo, top.d_drop_hi, col_, "v11_3b_top25_unique",
                 "3b · Top 25 signals — AUC the model loses without each (refitted)",
                 f"mean over {nF} test months with 95% t-interval · colour = family · money still here",
                 xlab="ΔAUC when removed and refitted")
    barh_err(lab_, top.d_perm, None, None, col_, "v11_3b_top25_reliance",
             "3b · Top 25 signals — AUC lost when the signal is shuffled in the test month",
             f"{PERM_REPS} shuffles per fold · colour = family", xlab="ΔAUC when shuffled")
    fig, ax = plt.subplots(figsize=(8.5, 6))
    xv = RANK.auc_alone.to_numpy(float)
    yv_ = (RANK.d_drop if RUN_DROP_ONE else RANK.d_perm).to_numpy(float)
    for f in FAMS:
        m = (RANK.family == f).to_numpy()
        ax.scatter(xv[m], yv_[m], s=26, color=FAMCOL.get(f, GREY), label=f, alpha=.85)
    for r, x_, y_ in zip(RANK.itertuples(), xv, yv_):
        if r.rank <= 12: ax.annotate(r.sig, (x_, y_), fontsize=6.5, xytext=(3, 2), textcoords="offset points")
    ax.axhline(0, color=INK, lw=.8); ax.axvline(0.5, color=INK, lw=.8, ls=":")
    ax.axvspan(0.5 - REDUNDANT_MIN, 0.5 + REDUNDANT_MIN, color="#F3F4F6", zorder=0)
    ax.set_xlabel("standalone AUC"); ax.set_ylabel("unique contribution (ΔAUC, refitted)" if RUN_DROP_ONE
                                                    else "reliance (ΔAUC, shuffled)")
    ax.legend(frameon=False, fontsize=7.5)
    _save(fig, "v11_3b_redundancy_map", "3b · Predictive alone vs needed by the model",
          "right and low = redundant (another signal carries it) · top = core · below 0 = the model is better without it")

In [ ]:
# =====================================================================
# 3c · FAMILIES DROPPED WHOLE + REDUCED MODELS, judged in dollars [OUTPUT BLOCK 5]
# =====================================================================
# Single-signal drops understate a family whose members cover for each other.
# Here each family leaves the model entirely; the refit is scored on the same
# test months and compared with the full model on the SHIP metric — capped
# dollars reached by the 1,000-a-month queue — with v10d's paired bootstrap.
def refit_score(cols, origins, tag):
    """Refit on `cols` for each origin (warm from the full fit), recalibrate,
    score; rows come back in SCF order for those origins."""
    praw_all, p_all, aucs = [], [], []
    for T in origins:
        fit_df, cal_df, te = fold_frames(T)
        sp = fit_spec(fit_df, cols, warm=SPEC[T])
        assert sp is not None and sp["converged"], f"{tag}: origin {T} did not converge"
        pr = predict_p(sp, te)
        kn = pav(predict_p(sp, cal_df), cal_df.y.to_numpy(), cal_df.sw.to_numpy())
        praw_all.append(pr); p_all.append(apply_iso(kn, pr)); aucs.append(auc(te.y.to_numpy(), pr))
    return np.concatenate(praw_all), np.concatenate(p_all), np.asarray(aucs)

t0 = time.time()
FAMF = SCF.copy()
_rows, _pair = [], []
_full_auc = FOLDS.set_index("origin").auc.reindex(ORIG).to_numpy()
for f in FAMS:
    cols_f = [c for c in COLS if FAM[sig_of(c)] != f]
    _, FAMF[f"p_no_{f}"], a = refit_score(cols_f, ORIG, f"no_{f}")
    _, tp = queue_tp(FAMF, f"no_{f}")
    d = _full_auc - a
    _rows.append(dict(family_dropped=f, signals=sum(1 for s in SIGS if FAM[s] == f),
                      cols=len(COLS) - len(cols_f), auc=a.mean(), d_auc=d.mean(),
                      d_auc_lo=d.mean() - tcrit(len(d)-1)*d.std(ddof=1)/np.sqrt(len(d)),
                      d_auc_hi=d.mean() + tcrit(len(d)-1)*d.std(ddof=1)/np.sqrt(len(d)),
                      folds_hurt=f"{int((d > 0).sum())} of {len(d)}",
                      departing_clients=len(tp), dollars=float(tp.bar_cap.sum())))
    _pair.append(dict(family_dropped=f, **paired(FAMF, "full", f"no_{f}")))
FAMT = pd.DataFrame(_rows)
FAMP = pd.DataFrame(_pair)
FAMT["dollars_lost"] = float(Q_TP.bar_cap.sum()) - FAMT.dollars
RES["top_family"] = str(FAMT.sort_values("dollars_lost", ascending=False).family_dropped.iloc[0])
RES["top_family_auc"] = str(FAMT.sort_values("d_auc", ascending=False).family_dropped.iloc[0])
disp(FAMT.assign(auc=FAMT.auc.round(4),
                 d_auc=[f"{f4(m)}  [{f4(lo)}, {f4(hi)}]" for m, lo, hi in
                        zip(FAMT.d_auc, FAMT.d_auc_lo, FAMT.d_auc_hi)],
                 dollars=FAMT.dollars.map(usd), dollars_lost=FAMT.dollars_lost.map(usd))
     .drop(columns=["d_auc_lo", "d_auc_hi"]),
     title=f"3c &middot; <b>Each family dropped whole and the model refitted.</b> Full model: "
           f"AUC {FOLDS.auc.mean():.4f}, {len(Q_TP)} departing clients, {usd(Q_TP.bar_cap.sum())} capped",
     save="v11_3c_family_drop")
disp(FAMP.assign(**{c: FAMP[c].map(usd) for c in ["difference", "ci_lo", "ci_hi", "month_ci_lo", "month_ci_hi"]},
                 p_first_ahead=FAMP.p_first_ahead.map(pctf)),
     title="3c &middot; <b>Does the full model reach more dollars than the model without the family?</b> "
           "Client bootstrap (95%), clients-to-flip, month by month (v10d §5). Positive = the family earns "
           "its place in dollars", save="v11_3c_family_paired")
bars(FAMT.assign(d=FAMT.d_auc), "family_dropped", ["d"], "v11_3c_family_auc",
     "3c · AUC lost when a family is removed and the model refitted",
     f"mean over {len(ORIG)} test months · money still here", fmt=f4, colors={"d": ACC2})
bars(FAMT.assign(d=FAMT.dollars_lost), "family_dropped", ["d"], "v11_3c_family_dollars",
     "3c · Capped dollars the queue stops reaching when a family is removed",
     "1,000 alerts a month · α = 0.5 · 7 test months", colors={"d": ACC})

# ── reduced models: rank on the early folds, test on the late ones ─────
def consensus_order(L):
    g = L.groupby("sig").agg(a=("auc_alone", "mean"), dp=("d_perm", "mean"), dd=("d_drop", "mean"))
    r = pd.concat([g.a.sub(0.5).abs().rank(ascending=False), g.dp.rank(ascending=False)]
                  + ([g.dd.rank(ascending=False)] if g.dd.notna().any() else []), axis=1).mean(axis=1)
    return list(r.sort_values().index)
RED = None
if len(ORIG) > RANK_SPLIT:
    early, late = ORIG[:RANK_SPLIT], ORIG[RANK_SPLIT:]
    order = consensus_order(LENS[LENS.origin.isin(early)])
    LATE = SCF[SCF.m_idx.isin(late)].copy()
    full_late = FOLDS.set_index("origin").auc.reindex(late).to_numpy()
    _rows, _pair = [], []
    for N in [n for n in TOPN_GRID if n < len(SIGS)]:
        cols_n = [c for s in order[:N] for c in PAIR[s]]
        _, LATE[f"p_top{N}"], a = refit_score(cols_n, late, f"top{N}")
        _, tp = queue_tp(LATE, f"top{N}")
        _rows.append(dict(model=f"top {N}", signals=N, cols=len(cols_n), auc=a.mean(),
                          gap_vs_all=full_late.mean() - a.mean(), departing_clients=len(tp),
                          dollars=float(tp.bar_cap.sum())))
        _pair.append(dict(model=f"top {N}", **paired(LATE, "full", f"top{N}")))
    _, tpf = queue_tp(LATE, "full")
    _rows.append(dict(model=f"all {len(SIGS)}", signals=len(SIGS), cols=len(COLS), auc=full_late.mean(),
                      gap_vs_all=0.0, departing_clients=len(tpf), dollars=float(tpf.bar_cap.sum())))
    RED = pd.DataFrame(_rows)
    if (RED.signals == 10).any():
        RES["top10_gap"] = float(RED.loc[RED.signals == 10, "gap_vs_all"].iloc[0])
    disp(RED.assign(auc=RED.auc.round(4), gap_vs_all=RED.gap_vs_all.map(f4), dollars=RED.dollars.map(usd)),
         title=f"3c &middot; <b>How few signals does the model need?</b> Signals ranked on the first "
               f"{len(early)} folds only; the reduced models are refitted and judged on the last "
               f"{len(late)} (no ranking leak). Top 10: {', '.join(order[:10])}",
         save="v11_3c_reduced_models")
    _rp = pd.DataFrame(_pair)
    disp(_rp.assign(**{c: _rp[c].map(usd) for c in ["difference", "ci_lo", "ci_hi", "month_ci_lo", "month_ci_hi"]},
                    p_first_ahead=_rp.p_first_ahead.map(pctf)),
         title="3c &middot; <b>All signals vs the reduced model, in capped dollars</b> — late folds only",
         save="v11_3c_reduced_paired")
else:
    print(f"  reduced-model test skipped: {len(ORIG)} folds ≤ RANK_SPLIT = {RANK_SPLIT}")
print(f"  §3c wall {time.time()-t0:,.0f}s")

In [ ]:
# =====================================================================
# 4 · REASON CODES — why this client is on the list          [OUTPUT BLOCK 6]
# =====================================================================
# The model is additive on the log-odds scale, so each alert's score splits
# exactly into per-signal pieces: beta × (x − training mean), value and flag
# summed. The three largest POSITIVE pieces are the reasons. They explain the
# model, not the client — a reason is "what moved the score", not a cause.
# "higher / lower than typical" compares the client with the training average of
# that feature; for ld_ features that is already a change against the client's
# own months t-6..t-4, relative to its peer group (04 §1).
def contributions(sp, df):
    X = df[sp["cols"]].to_numpy(np.float64)
    C = (X - sp["mu"]) * sp["beta"]
    by = pd.DataFrame(C, columns=sp["cols"], index=df.index)
    return by.T.groupby(by.columns.map(sig_of)).sum().T        # one column per signal

def reason_text(sig, row, sp):
    ld, md = f"ld_{sig}", f"md_{sig}"
    mu = dict(zip(sp["cols"], sp["mu"]))
    if md in row.index and row[md] >= 0.5:
        return f"{describe(sig)}: not observed"
    if ld in row.index and ld in mu:
        return f"{describe(sig)}: {'higher' if row[ld] > mu[ld] else 'lower'} than typical"
    return describe(sig)

_parts = []
for T in ORIG:
    qa = Q_FIRST[Q_FIRST.m_idx == T][["cust_pwr_id", "m_idx", "y", "p_full", "bar_now", "bar_med12"]]
    if qa.empty: continue
    te = TE[T]
    x = qa.merge(te[["cust_pwr_id"] + COLS].drop_duplicates("cust_pwr_id"), on="cust_pwr_id", how="left")
    sp = SPEC[T]
    C = contributions(sp, x)
    # additivity self-check: pieces + intercept + Σ beta·mu reproduce the logit
    eta = sp["b0"] + x[sp["cols"]].to_numpy(np.float64) @ sp["beta"]
    assert np.allclose(C.sum(axis=1).to_numpy() + sp["b0"] + float(sp["mu"] @ sp["beta"]), eta, atol=1e-8), \
        "reason codes do not add up to the score"
    top = np.argsort(-C.to_numpy(), axis=1)[:, :3]
    names = C.columns.to_numpy()
    for i in range(3):
        x[f"r{i+1}_sig"] = names[top[:, i]]
        x[f"r{i+1}_pts"] = C.to_numpy()[np.arange(len(C)), top[:, i]]
        x[f"reason_{i+1}"] = [reason_text(s, x.iloc[j], sp) if pts > 0 else "—"
                              for j, (s, pts) in enumerate(zip(x[f"r{i+1}_sig"], x[f"r{i+1}_pts"]))]
    x["logit_above_mean"] = C.sum(axis=1).to_numpy()
    _parts.append(x.drop(columns=COLS))
REASONS = pd.concat(_parts, ignore_index=True)
REASONS["top_family"] = REASONS.r1_sig.map(FAM)

def _freq(d, nm):
    r1 = d.r1_sig.value_counts(normalize=True).rename(f"{nm}: reason #1")
    r3 = (pd.concat([d.r1_sig, d.r2_sig, d.r3_sig]).value_counts()/len(d)).rename(f"{nm}: in top 3")
    return pd.concat([r1, r3], axis=1)
FREQ = pd.concat([_freq(REASONS, "all alerts"), _freq(REASONS[REASONS.y == 1], "departing")], axis=1).fillna(0)
FREQ = FREQ.sort_values("all alerts: in top 3", ascending=False)
FREQ.insert(0, "description", FREQ.index.map(describe)); FREQ.insert(1, "family", FREQ.index.map(FAM))
FREQ.insert(2, "tier", FREQ.index.map(RANK.set_index("sig").tier))
disp(FREQ.head(25).reset_index(names="signal").assign(
        **{c: FREQ.head(25)[c].map(pctf).to_numpy() for c in FREQ.columns[3:]}),
     title=f"4a &middot; <b>The reasons RMs would read.</b> {len(REASONS):,} first alerts over "
           f"{len(ORIG)} months; how often each signal is the top reason, and how often it is in the top "
           f"three — for all alerts and for the {int(REASONS.y.sum()):,} that were genuinely leaving",
     save="v11_4a_reason_frequency")
_rf = (REASONS.groupby("top_family").agg(alerts=("cust_pwr_id", "size"), departing=("y", "sum"))
       .reindex(FAMS).fillna(0))
_rf["alert_share"] = _rf.alerts/_rf.alerts.sum(); _rf["precision"] = _rf.departing/_rf.alerts.replace(0, np.nan)
disp(_rf.reset_index().assign(alert_share=_rf.alert_share.map(pctf).to_numpy(),
                              precision=_rf.precision.map(pctf).to_numpy()),
     title="4b &middot; <b>Which family supplies the top reason</b>, and how often those alerts were right",
     save="v11_4b_reason_family")
bars(_rf.reset_index().rename(columns={"top_family": "family"}).assign(s=_rf.alert_share.to_numpy()),
     "family", ["s"], "v11_4b_reason_family", "4b · Share of alerts whose top reason comes from each family",
     "first alerts · 1,000 a month · money still here", fmt=pctf, colors={"s": ACC2})

# the latest month's list, as an RM would receive it
T_LAST = max(ORIG)
L = REASONS[REASONS.m_idx == T_LAST].copy()
L["score"] = qscore(L.p_full, L.bar_med12)
L = L.sort_values("score", ascending=False)
_cols = ["client", "p_full", "bar_med12", "bar_now", "reason_1", "reason_2", "reason_3", "y"]
disp(L.head(25).assign(client=L.head(25).cust_pwr_id.map(cid), p_full=L.head(25).p_full.map(pctf),
                       bar_med12=L.head(25).bar_med12.map(usd), bar_now=L.head(25).bar_now.map(usd))[_cols],
     title=f"4c &middot; <b>Month {T_LAST}: the top 25 new alerts with their reasons.</b> y = left within "
           f"{PRIMARY_H} months (known only in back-test). Full ids: v11_4c_rm_list_FULL_IDS.csv")
L.to_csv(OUT_DIR / "v11_4c_rm_list_FULL_IDS.csv", index=False)

## 5 · Leaving because the business is shrinking, or moving the banking elsewhere?

**The RM's decision.** *Contraction* — revenue falling, payroll shrinking, a wind-down — calls for credit,
working capital, restructuring, patience. *Displacement* — the business is fine and its banking is moving —
calls for competing on price, earnings credit, service and product. Getting it backwards is expensive in
both directions: a line of credit offered to a client already onboarding elsewhere, or an ECR match offered
to a client whose sales halved.

### The evidence, per client, around its event month `e`

Windows in months relative to `e`: **base** −12…−9 (normal trading), **drain** −8…−1 (the money leaves —
v9/v10d), **post** 0…5 (after the last account closes).

| Indicator | Definition | Reads as |
|---|---|---|
| `e_self` | outbound to the client's **own name** at a non-PNC institution in the drain window, **above the client's own base rate**, as a share of its normal balance | displacement |
| `cont_else` | of the client's **base-window internal trading partners** (other PNC customers, excluding its own entities and its relationship), the $-weighted share that pay — or are paid by — the client's name **outside PNC** in the post window | displacement — relationships survived the move |
| `persist_any` | the same share, counting continuation anywhere (inside PNC or outside) | its absence (`dead`) reads as contraction |
| `e_new` | share of drain-window outbound institution dollars going to institutions the client **never used before** rel_m −9 | a new bank — but a sale or a wind-down to one creditor looks the same, so it is kept as its own class |
| `e_sib` | drain-window outflow to the client's own other PNC entities, relationship siblings or a same-named PNC customer, as a share of normal balance | not attrition — a re-key |

Names are matched on a **compact key** (upper case → alphanumerics → legal suffixes dropped → first 12
characters, ≥ 8 to count). Keys on the v8 generic-name registry never count; a key shared by two PNC
customers is not used to attribute partners' flows. ACH truncates names — the 12-character key is what
survives truncation.

### Calibration against stayers
Each client who never left gets a **pseudo-event** drawn from the attriters' event-month distribution, and
the same windows are built around it. Each threshold is the larger of a fixed floor and the stayers' 95th
percentile — so **every indicator fires for at most 5% of clients who did not leave**. The `dead` threshold is
the smaller of 0.10 and the stayers' 5th percentile of `persist_any`.

### Classes (A attriters with a normal balance ≥ $10k at rel_m −12; the rest are `NOT_TYPEABLE`)

| Class | Rule | RM play |
|---|---|---|
| `INTERNAL` | `e_sib` fires, no displacement evidence | verify — likely a re-key, not a loss |
| `DISPLACEMENT` | `e_self` or `cont_else` fires, relationships not dead | compete |
| `MIXED` | displacement evidence **and** dead relationships; or dead + a dominant new bank | discover |
| `CONTRACTION` | relationships dead, no displacement evidence, **name-match recall ≥ 50%** | support |
| `LEANS_CONTRACTION` | as above but recall < 50% (or unmeasurable) — "dead" may be a missed match | support, softly |
| `MOVED_TO_NEW_BANK` | only `e_new` fires | discover (leans compete) |
| `NO_DIRECT_EVIDENCE` | nothing fires, or relationship survival not observable | discover |

### Why a recall proxy gates contraction
A missed name match and a relationship that genuinely stopped look identical. Clients that moved at least
half their normal balance **to their own name** elsewhere are displaced beyond doubt; the share of them whose
partners are seen continuing (`cont_else > 0`) estimates how often the matcher *can* see survival when it
exists. If that is low, "dead" is mostly blindness, and contraction is only a lean.

### Limits, stated up front
Relationship survival is only visible where the partner is itself a PNC customer — coverage is measured
(§6), not assumed. Everything here is a **hypothesis about why**, built from payment behaviour; it becomes a
finding only when RMs confirm a sample (§7 writes it) and, ultimately, when win/loss notes are recorded.

In [ ]:
# =====================================================================
# 5a · EVIDENCE BUILD — one Spark job per payment month, resumable [OUTPUT BLOCK 7]
# =====================================================================
# Writes, per payment month, under attrition_v11/evidence/<kind>/ym=YYYY-MM:
#   cm   client × month: own external flows (out / in / to institutions / to its
#        OWN NAME elsewhere / wire / largest payment / breadth) + internal flows
#        split own-entity & sibling vs other PNC customers
#   dest client × month × destination institution (top DEST_TOP_N by $)
#   int  client × month × internal PNC partner (top INT_TOP_N by $), tagged
#        own / sib / same_name / other
#   sh   "shadow": other PNC customers paying — or paid by — a counterparty
#        whose name key is the client's, on non-card rails (top SH_TOP_N)
#   mdm  the client's own MDM ids seen that month
# One job per month: a single unfiltered pass killed the SparkContext (09 §2).
from pyspark.sql import functions as F, Window
from pyspark import StorageLevel
EV_ROOT = hp("v11", "evidence")
EV_KINDS = ["cm", "dest", "int", "sh", "mdm"]
def ev_path(kind, ym): return f"{EV_ROOT}/{kind}/ym={ym}"
def m_to_ym(m):
    y = (m - 1)//12
    return f"{y:04d}-{m - 12*y:02d}"
def month_bounds(m):
    y = (m - 1)//12; mo = m - 12*y
    return f"{y:04d}-{mo:02d}-01", f"{y:04d}-{mo:02d}-{calendar.monthrange(y, mo)[1]:02d}"
PAY_MONTHS = list(range(M_MIN, M_MAX + 1))
assert m_to_ym(M_MIN) == DATE_START[:7], "m_idx ↔ calendar mapping broken"

# ── the name key ──────────────────────────────────────────────────────
_LEGAL = ["THE", "INC", "LLC", "LTD", "CORP", "CO", "COMPANY", "LP", "LLP", "PLC", "PC", "PA", "TRUST"]
_LEGAL_RE = r" (?:" + "|".join(_LEGAL) + r")(?= )"
def compact_key(col):
    """UPPER → non-alphanumerics to spaces → legal words dropped (word-bounded,
    so 'CORPORATE' survives) → spaces removed → first KEY_LEN chars; NULL when
    shorter than MIN_KEY_CHARS (MOM, DAD, CHECKING never key anything)."""
    s = F.upper(F.coalesce(col, F.lit("")))
    s = F.regexp_replace(s, r"[^A-Z0-9]+", " ")
    s = F.concat(F.lit(" "), s, F.lit(" "))
    s = F.regexp_replace(s, r" L L C(?= )", "")
    s = F.regexp_replace(s, _LEGAL_RE, "")
    s = F.substring(F.regexp_replace(s, " ", ""), 1, KEY_LEN)
    return F.when(F.length(s) >= MIN_KEY_CHARS, s)
_kt = spark.createDataFrame([(1, "The Acme Tool Co., Inc."), (2, "ACME TOOL CO"), (3, "MOM"),
                             (4, "Acme Tool Company LLC"), (5, "A.B.C. Holdings L L C"),
                             (6, "Corporate Supply Partners of Ohio"), (7, None)], "i int, nm string")
_got = {r.i: r.k for r in _kt.select("i", compact_key(F.col("nm")).alias("k")).collect()}
_want = {1: "ACMETOOL", 2: "ACMETOOL", 3: None, 4: "ACMETOOL", 5: "ABCHOLDINGS", 6: "CORPORATESUP", 7: None}
assert _got == _want, f"compact_key self-check failed: {_got}"
def _pop(c):
    return F.coalesce(F.col(c).isNotNull() & (F.trim(F.col(c)) != "") &
                      (~F.upper(F.trim(F.col(c))).isin("NULL", "NA", "N/A", "UNKNOWN", "-1")), F.lit(False))

# ── account map + customer dimension (cached on HDFS) ─────────────────
AMAP_P, CDIM_P = hp("v11", "amap"), hp("v11", "cust_dim")
if REBUILD_EVIDENCE or not (hdfs_exists(AMAP_P + "/_SUCCESS") and hdfs_exists(CDIM_P + "/_SUCCESS")):
    dep = spark.table(DEP_TBL).filter(F.col("edw_tda_load_dt").between(DATE_START, DATE_END))
    wa = Window.partitionBy("acct_full_acct_id").orderBy(F.desc("edw_tda_load_dt"))
    (dep.select("acct_full_acct_id", "cust_pwr_id", "edw_tda_load_dt")
        .filter(F.col("cust_pwr_id").isNotNull() & F.col("acct_full_acct_id").isNotNull())
        .withColumn("rn", F.row_number().over(wa)).filter("rn = 1")
        .select(F.col("acct_full_acct_id").cast("string").alias("acct"),
                F.col("cust_pwr_id").cast("string").alias("cust_pwr_id"))
        .write.mode("overwrite").parquet(AMAP_P))
    wc = Window.partitionBy("cust_pwr_id").orderBy(F.col("cust_name").isNull(), F.desc("edw_tda_load_dt"))
    (dep.select("cust_pwr_id", "cust_name", "rltn_pwr_id", "edw_tda_load_dt")
        .filter(F.col("cust_pwr_id").isNotNull())
        .withColumn("rn", F.row_number().over(wc)).filter("rn = 1")
        .select(F.col("cust_pwr_id").cast("string").alias("cust_pwr_id"), "cust_name",
                F.col("rltn_pwr_id").cast("string").alias("rltn_pwr_id"))
        .withColumn("own_key", compact_key(F.col("cust_name")))
        .write.mode("overwrite").parquet(CDIM_P))
amap = spark.read.parquet(AMAP_P).persist(StorageLevel.DISK_ONLY)
cdim = spark.read.parquet(CDIM_P).persist(StorageLevel.DISK_ONLY)
_na = amap.count()
assert _na == amap.select("acct").distinct().count(), "account map is not 1:1 — it would fan out volume"

# generic registry (v8) and keys shared across relationships
_gen = spark.read.parquet(hp("v8", "generic_names"))
_gcol = (next((c for c, t in _gen.dtypes if t == "string" and ("name" in c.lower() or "key" in c.lower())), None)
         or next(c for c, t in _gen.dtypes if t == "string"))
GENK = (_gen.select(compact_key(F.col(_gcol)).alias("own_key")).filter(F.col("own_key").isNotNull())
        .distinct().withColumn("generic", F.lit(True)))
AMBK = (cdim.filter(F.col("own_key").isNotNull()).groupBy("own_key")
        .agg(F.countDistinct(F.coalesce("rltn_pwr_id", "cust_pwr_id")).alias("_nr"))
        .filter(F.col("_nr") > 1).select("own_key", F.lit(True).alias("ambiguous")))
TGT = to_sdf(LABS[["cust_pwr_id"]].astype(str))
TK = (cdim.join(TGT, "cust_pwr_id").join(GENK, "own_key", "left").join(AMBK, "own_key", "left")
      .withColumn("usable_self", F.col("own_key").isNotNull() & ~F.coalesce("generic", F.lit(False)))
      .withColumn("usable_shadow", F.col("usable_self") & ~F.coalesce("ambiguous", F.lit(False)))
      .select("cust_pwr_id", "rltn_pwr_id", "own_key", "usable_self", "usable_shadow")
      .persist(StorageLevel.DISK_ONLY))
SHK = (TK.filter("usable_shadow").select(F.col("own_key").alias("ckey"), F.col("cust_pwr_id").alias("tgt"),
                                         F.col("rltn_pwr_id").alias("tgt_rltn")))
PR = cdim.select(F.col("cust_pwr_id").alias("partner_cust"), F.col("rltn_pwr_id").alias("partner_rltn"))
_tk = TK.agg(F.count(F.lit(1)).alias("n"), F.sum(F.col("usable_self").cast("int")).alias("s"),
             F.sum(F.col("usable_shadow").cast("int")).alias("h")).collect()[0]

def _topn(df, part, n, order="amt"):
    w = Window.partitionBy(*part).orderBy(F.desc(order))
    return df.withColumn("_r", F.row_number().over(w)).filter(F.col("_r") <= n).drop("_r")

def build_month(m):
    ym = m_to_ym(m); lo, hi = month_bounds(m)
    p = (spark.table(PAY_TBL).filter(F.col("trans_dt").between(lo, hi))
         .select(F.col("trans_amt").cast("double").alias("amt"), "payment_rail", "mdm_id_pays",
                 "mdm_id_receives", "pnc_dep_acct_pays", "pnc_dep_acct_receives", "cpty_name",
                 "cpty_fin_entity_name", "customer_name_pays", "customer_name_receives")
         .filter(F.col("amt") > 0))
    one_out = F.col("mdm_id_pays").isNotNull() & F.col("mdm_id_receives").isNull()
    one_in = F.col("mdm_id_receives").isNotNull() & F.col("mdm_id_pays").isNull()
    # (1) the client's own external flows
    sides = (p.filter(one_out | one_in)
             .select(F.when(one_out, F.lit("out")).otherwise(F.lit("in")).alias("dir"),
                     F.when(one_out, F.col("pnc_dep_acct_pays")).otherwise(F.col("pnc_dep_acct_receives"))
                      .cast("string").alias("acct"),
                     F.when(one_out, F.col("mdm_id_pays")).otherwise(F.col("mdm_id_receives"))
                      .cast("string").alias("mdm"),
                     "amt", "payment_rail", "cpty_name", _pop("cpty_fin_entity_name").alias("hf"),
                     F.upper(F.trim(F.col("cpty_fin_entity_name"))).alias("fin"))
             .join(F.broadcast(amap), "acct").join(F.broadcast(TK), "cust_pwr_id")
             .withColumn("ckey", compact_key(F.col("cpty_name")))
             .withColumn("is_self", F.coalesce(F.col("usable_self") & (F.col("ckey") == F.col("own_key")),
                                               F.lit(False)))
             .persist(StorageLevel.DISK_ONLY))
    o, i = F.col("dir") == "out", F.col("dir") == "in"
    def S(c, v="amt"): return F.sum(F.when(c, F.col(v)).otherwise(F.lit(0.0)))
    def N(c): return F.sum(F.when(c, F.lit(1)).otherwise(F.lit(0)))
    cpk = F.coalesce(F.col("ckey"), F.upper(F.trim(F.col("cpty_name"))))
    cm_x = sides.groupBy("cust_pwr_id").agg(
        S(o).alias("out_amt"), N(o).alias("out_n"), S(i).alias("in_amt"), N(i).alias("in_n"),
        S(o & F.col("hf")).alias("out_fin_amt"),
        S(o & F.col("is_self") & F.col("hf")).alias("self_out_amt"),
        N(o & F.col("is_self") & F.col("hf")).alias("self_out_n"),
        S(i & F.col("is_self")).alias("self_in_amt"),
        S(o & (F.col("payment_rail") == "WIRE")).alias("wire_out_amt"),
        F.max(F.when(o, F.col("amt"))).alias("max_out_txn"),
        F.countDistinct(F.when(o, cpk)).alias("n_cpty_out"),
        F.countDistinct(F.when(i, cpk)).alias("n_cpty_in"))
    # (2) internal PNC-to-PNC flows, seen from the client's side
    intr = (p.filter(F.col("mdm_id_pays").isNotNull() & F.col("mdm_id_receives").isNotNull())
            .select("amt", F.col("mdm_id_pays").cast("string").alias("mdm_p"),
                    F.col("mdm_id_receives").cast("string").alias("mdm_r"),
                    F.col("pnc_dep_acct_pays").cast("string").alias("acct_p"),
                    F.col("pnc_dep_acct_receives").cast("string").alias("acct_r"),
                    F.col("customer_name_pays").alias("nm_p"), F.col("customer_name_receives").alias("nm_r"))
            .join(F.broadcast(amap.select(F.col("acct").alias("acct_p"), F.col("cust_pwr_id").alias("cust_p"))),
                  "acct_p", "left")
            .join(F.broadcast(amap.select(F.col("acct").alias("acct_r"), F.col("cust_pwr_id").alias("cust_r"))),
                  "acct_r", "left"))
    _a = intr.filter(F.col("cust_p").isNotNull()).select(
        F.col("cust_p").alias("cust_pwr_id"), F.lit("out").alias("dir"), F.col("mdm_p").alias("own_mdm"),
        F.col("mdm_r").alias("partner_mdm"), F.col("cust_r").alias("partner_cust"),
        F.col("nm_r").alias("partner_nm"), "amt")
    _b = intr.filter(F.col("cust_r").isNotNull()).select(
        F.col("cust_r").alias("cust_pwr_id"), F.lit("in").alias("dir"), F.col("mdm_r").alias("own_mdm"),
        F.col("mdm_p").alias("partner_mdm"), F.col("cust_p").alias("partner_cust"),
        F.col("nm_p").alias("partner_nm"), "amt")
    ic = (_a.unionByName(_b).join(F.broadcast(TK), "cust_pwr_id").join(F.broadcast(PR), "partner_cust", "left")
          .withColumn("pkey", compact_key(F.col("partner_nm")))
          .withColumn("rel", F.when(F.col("partner_cust") == F.col("cust_pwr_id"), F.lit("self_acct"))
                               .when(F.col("partner_mdm") == F.col("own_mdm"), F.lit("own"))
                               .when(F.col("partner_rltn") == F.col("rltn_pwr_id"), F.lit("sib"))
                               .when(F.col("usable_self") & (F.col("pkey") == F.col("own_key")), F.lit("same_name"))
                               .otherwise(F.lit("other")))
          .filter(F.col("rel") != "self_acct")
          .persist(StorageLevel.DISK_ONLY))
    sibc = F.col("rel").isin("own", "sib", "same_name")
    cm_i = ic.groupBy("cust_pwr_id").agg(
        S(o).alias("int_out_amt"), S(i).alias("int_in_amt"),
        S(o & sibc).alias("int_sib_out_amt"), S(i & sibc).alias("int_sib_in_amt"),
        S(o & ~sibc).alias("int_oth_out_amt"), S(i & ~sibc).alias("int_oth_in_amt"))
    cm = cm_x.join(cm_i, "cust_pwr_id", "full").na.fill(0.0).withColumn("m_idx", F.lit(m))
    cm.write.mode("overwrite").parquet(ev_path("cm", ym))
    dest = (sides.filter(o & F.col("hf")).groupBy("cust_pwr_id", "fin")
            .agg(F.sum("amt").alias("amt"), F.count(F.lit(1)).alias("n"),
                 S(F.col("is_self")).alias("self_amt")))
    _topn(dest, ["cust_pwr_id"], DEST_TOP_N).withColumn("m_idx", F.lit(m)) \
        .write.mode("overwrite").parquet(ev_path("dest", ym))
    it = (ic.groupBy("cust_pwr_id", "dir", "partner_mdm", "rel")
          .agg(F.sum("amt").alias("amt"), F.count(F.lit(1)).alias("n"), F.first("partner_cust").alias("partner_cust")))
    _topn(it, ["cust_pwr_id", "dir"], INT_TOP_N).withColumn("m_idx", F.lit(m)) \
        .write.mode("overwrite").parquet(ev_path("int", ym))
    (sides.select("cust_pwr_id", F.col("mdm").alias("own_mdm"))
          .unionByName(ic.select("cust_pwr_id", "own_mdm")).filter(F.col("own_mdm").isNotNull()).distinct()
          .withColumn("m_idx", F.lit(m)).write.mode("overwrite").parquet(ev_path("mdm", ym)))
    # (3) shadow: another PNC customer's external flow whose counterparty key is the client's.
    #     Direction is from the CLIENT's side: the partner paying the key = the client receiving.
    sh = (p.filter((one_out | one_in) & F.col("payment_rail").isin(SHADOW_RAILS))
          .select(F.when(one_out, F.lit("in")).otherwise(F.lit("out")).alias("dir"),
                  F.when(one_out, F.col("pnc_dep_acct_pays")).otherwise(F.col("pnc_dep_acct_receives"))
                   .cast("string").alias("acct"),
                  F.when(one_out, F.col("mdm_id_pays")).otherwise(F.col("mdm_id_receives"))
                   .cast("string").alias("partner_mdm"),
                  "amt", _pop("cpty_fin_entity_name").alias("hf"), compact_key(F.col("cpty_name")).alias("ckey"))
          .filter(F.col("ckey").isNotNull())
          .join(F.broadcast(SHK), "ckey")
          .join(F.broadcast(amap.select("acct", F.col("cust_pwr_id").alias("partner_cust"))), "acct", "left")
          .join(F.broadcast(PR), "partner_cust", "left")
          .filter(~F.coalesce((F.col("partner_cust") == F.col("tgt")) |
                              (F.col("partner_rltn") == F.col("tgt_rltn")), F.lit(False)))
          .groupBy(F.col("tgt").alias("cust_pwr_id"), "partner_mdm", "dir")
          .agg(F.sum("amt").alias("amt"), F.count(F.lit(1)).alias("n"),
               F.max(F.col("hf").cast("int")).alias("has_fin")))
    _topn(sh, ["cust_pwr_id"], SH_TOP_N).withColumn("m_idx", F.lit(m)) \
        .write.mode("overwrite").parquet(ev_path("sh", ym))
    sides.unpersist(); ic.unpersist()

t0 = time.time(); built, skipped = [], []
if RUN_EVIDENCE_BUILD:
    for m in PAY_MONTHS:
        ym = m_to_ym(m)
        if not REBUILD_EVIDENCE and all(hdfs_exists(ev_path(k, ym) + "/_SUCCESS") for k in EV_KINDS):
            skipped.append(ym); continue
        t1 = time.time(); build_month(m); built.append(ym)
        print(f"  evidence {ym}: {time.time()-t1:,.0f}s  (elapsed {(time.time()-t0)/60:,.1f} min)")
_have = [m_to_ym(m) for m in PAY_MONTHS if all(hdfs_exists(ev_path(k, m_to_ym(m)) + "/_SUCCESS") for k in EV_KINDS)]
assert len(_have) == len(PAY_MONTHS), (f"evidence missing for {len(PAY_MONTHS) - len(_have)} months — "
                                       f"set RUN_EVIDENCE_BUILD = True and rerun this cell")
EVIDENCE_REBUILT = bool(built)
kv([("payment months", f"{len(PAY_MONTHS)} · built this run {len(built)} · already on HDFS {len(skipped)}"),
    ("account map", f"{_na:,} accounts, 1:1"),
    ("target clients (all labelled)", f"{int(_tk['n']):,} · usable own-name key {int(_tk['s']):,} "
                                      f"({pctf(_tk['s']/max(_tk['n'], 1))}) · usable for partner matching "
                                      f"{int(_tk['h']):,} ({pctf(_tk['h']/max(_tk['n'], 1))})"),
    ("generic registry column used", _gcol),
    ("name-key self-check", "passed (7 cases incl. legal suffixes, word boundaries, short names)"),
    ("shadow rails", ", ".join(SHADOW_RAILS) + " — card descriptors are merchant strings, excluded"),
    ("wall", f"{(time.time()-t0)/60:,.1f} min")],
   title="5a &middot; <b>Evidence build</b>", save="v11_5a_build")

In [ ]:
# =====================================================================
# 5b · EVIDENCE PER CLIENT AROUND ITS EVENT + EVIDENCE-TO-DATE PANEL [OUTPUT BLOCK 8]
# =====================================================================
t0 = time.time()
EVR = {k: spark.read.parquet(f"{EV_ROOT}/{k}").drop("ym") for k in EV_KINDS}

# ── events: A, B-only, and stayers at pseudo-events ───────────────────
E = LABS.copy()
E["grp"] = np.select([E.qA.notna(), E.qB.notna()], ["A", "B_only"], "CTRL")
E["e"] = np.where(E.qA.notna(), E.qA, E.qB)
_a_ev = np.sort(E.loc[E.grp == "A", "e"].astype(int).to_numpy())
assert len(_a_ev) > 0 and _a_ev.min() >= M_MIN and _a_ev.max() <= M_MAX + 1, \
    f"q_A_full_exit is not an absolute m_idx in [{M_MIN}, {M_MAX + 1}]"
_h = pd.util.hash_pandas_object(E.cust_pwr_id + f"|{SEED}", index=False).to_numpy() % len(_a_ev)
E.loc[E.grp == "CTRL", "e"] = _a_ev[_h[(E.grp == "CTRL").to_numpy()]]
E["e"] = E.e.astype(int)
EVT = to_sdf(E[["cust_pwr_id", "grp", "e"]].astype({"e": "int64"}))

def _win(c, w): return F.col(c).between(w[0], w[1])
def WS(v, w): return F.sum(F.when(_win("rel", w), F.col(v)).otherwise(F.lit(0.0)))
REL_LO, REL_HI = BASE_WIN[0], POST_WIN[1]

# own flows by window
cmw = (EVR["cm"].join(F.broadcast(EVT), "cust_pwr_id")
       .withColumn("rel", F.col("m_idx") - F.col("e")).filter(F.col("rel").between(REL_LO, REL_HI)))
FLOW = cmw.groupBy("cust_pwr_id").agg(
    *[WS(v, w).alias(f"{nm}_{v}") for nm, w in [("base", BASE_WIN), ("drain", DRAIN_WIN), ("post", POST_WIN)]
      for v in ["out_amt", "out_n", "in_amt", "out_fin_amt", "self_out_amt", "self_in_amt", "wire_out_amt",
                "int_sib_out_amt", "int_out_amt", "n_cpty_out"]],
    F.max(F.when(_win("rel", DRAIN_WIN), F.col("max_out_txn"))).alias("drain_max_out"))

# balance: normal balance at rel -12, trajectory for drain speed, post rebound
balw = (BAL.select("cust_pwr_id", "m_idx", "bal_now", "bal_med12").join(F.broadcast(EVT), "cust_pwr_id")
        .withColumn("rel", F.col("m_idx") - F.col("e")).filter(F.col("rel").between(REL_LO, REL_HI)))
BALW = _dec(balw.groupBy("cust_pwr_id").pivot("rel", list(range(REL_LO, REL_HI + 1)))
            .agg(F.first("bal_now"))).toPandas()
_norm = _dec(balw.filter(F.col("rel") == BASE_WIN[0]).select("cust_pwr_id", F.col("bal_med12").alias("norm"))).toPandas()

# new institutions: first month each client paid each institution (top-N lists)
dst = EVR["dest"]
first_seen = dst.groupBy("cust_pwr_id", "fin").agg(F.min("m_idx").alias("fs"))
NEWB = (dst.join(F.broadcast(EVT), "cust_pwr_id").join(first_seen, ["cust_pwr_id", "fin"])
        .withColumn("rel", F.col("m_idx") - F.col("e")).filter(_win("rel", DRAIN_WIN))
        .groupBy("cust_pwr_id").agg(
            F.sum("amt").alias("drain_dest_amt"),
            F.sum(F.when(F.col("fs") > F.col("e") + KNOWN_DEST_END, F.col("amt")).otherwise(0.0)).alias("drain_new_amt"),
            F.sum(F.when(F.col("fs") > F.col("e") + KNOWN_DEST_END, F.col("self_amt")).otherwise(0.0)).alias("drain_new_self_amt")))

# relationship survival: base-window internal partners ("other" only), then who is still trading in post
OWN = EVR["mdm"].select("cust_pwr_id", F.col("own_mdm").alias("partner_mdm")).distinct().withColumn("_own", F.lit(1))
iw = (EVR["int"].withColumnRenamed("rel", "tag").join(F.broadcast(EVT), "cust_pwr_id")
      .withColumn("rel", F.col("m_idx") - F.col("e")))
BASEP = (iw.filter(_win("rel", BASE_WIN) & (F.col("tag") == "other"))
         .join(OWN, ["cust_pwr_id", "partner_mdm"], "left_anti")
         .groupBy("cust_pwr_id", "partner_mdm").agg(F.sum("amt").alias("base_amt")))
POST_INT = (iw.filter(_win("rel", POST_WIN) & (F.col("tag") == "other"))
            .select("cust_pwr_id", "partner_mdm").distinct().withColumn("in_pnc", F.lit(1)))
POST_SH = (EVR["sh"].join(F.broadcast(EVT), "cust_pwr_id").withColumn("rel", F.col("m_idx") - F.col("e"))
           .filter(_win("rel", POST_WIN)).join(OWN, ["cust_pwr_id", "partner_mdm"], "left_anti")
           .groupBy("cust_pwr_id", "partner_mdm").agg(F.max("has_fin").alias("sh_fin"))
           .withColumn("elsewhere", F.lit(1)))
CONT = (BASEP.join(POST_INT, ["cust_pwr_id", "partner_mdm"], "left")
        .join(POST_SH, ["cust_pwr_id", "partner_mdm"], "left")
        .groupBy("cust_pwr_id").agg(
            F.count(F.lit(1)).alias("base_partners"), F.sum("base_amt").alias("base_partner_amt"),
            F.sum(F.when(F.col("elsewhere") == 1, F.col("base_amt")).otherwise(0.0)).alias("cont_else_amt"),
            F.sum(F.when((F.col("elsewhere") == 1) | (F.col("in_pnc") == 1), F.col("base_amt")).otherwise(0.0))
             .alias("persist_amt"),
            F.sum(F.when(F.col("elsewhere") == 1, 1).otherwise(0)).alias("cont_else_partners")))
# post-window shadow volume from ANY partner (for the MOVED/continuation narrative)
SHV = (EVR["sh"].join(F.broadcast(EVT), "cust_pwr_id").withColumn("rel", F.col("m_idx") - F.col("e"))
       .join(OWN, ["cust_pwr_id", "partner_mdm"], "left_anti")
       .groupBy("cust_pwr_id").agg(WS("amt", BASE_WIN).alias("base_sh_amt"), WS("amt", POST_WIN).alias("post_sh_amt")))

EVID = E[["cust_pwr_id", "grp", "e"]].copy()
for part in [FLOW, NEWB, CONT, SHV]:
    EVID = EVID.merge(_dec(part).toPandas(), on="cust_pwr_id", how="left")
EVID = EVID.merge(_norm, on="cust_pwr_id", how="left")
EVID = EVID.merge(TK.select("cust_pwr_id", "usable_self", "usable_shadow").toPandas(), on="cust_pwr_id", how="left")
for c in EVID.columns:
    if c.startswith(("base_", "drain_", "post_", "cont_", "persist_")) and c != "drain_max_out":
        EVID[c] = pd.to_numeric(EVID[c], errors="coerce").fillna(0.0)
EVID["usable_self"] = EVID.usable_self.fillna(False).astype(bool)
EVID["usable_shadow"] = EVID.usable_shadow.fillna(False).astype(bool)

# ── indicators ────────────────────────────────────────────────────────
nb, nd = BASE_WIN[1] - BASE_WIN[0] + 1, DRAIN_WIN[1] - DRAIN_WIN[0] + 1
normc = np.maximum(pd.to_numeric(EVID.norm, errors="coerce").fillna(0.0).to_numpy(), 1.0)
EVID["obs_base"] = (EVID.e + BASE_WIN[0] >= M_MIN)
EVID["post_months"] = np.clip(M_MAX - EVID.e + 1, 0, POST_WIN[1] - POST_WIN[0] + 1)
EVID["typeable"] = EVID.obs_base & (pd.to_numeric(EVID.norm, errors="coerce").fillna(0) >= TYPE_MIN_NORM)
EVID["e_self"] = np.maximum(EVID.drain_self_out_amt - EVID.base_self_out_amt*nd/nb, 0.0)/normc
EVID["moved_self"] = EVID.drain_self_out_amt/normc
EVID["e_sib"] = np.maximum(EVID.drain_int_sib_out_amt - EVID.base_int_sib_out_amt*nd/nb, 0.0)/normc
_new_ok = (EVID.e + KNOWN_DEST_END - M_MIN + 1 >= NEW_BLANK_M) & (EVID.drain_dest_amt > 0)
EVID["e_new"] = np.where(_new_ok, sdiv(EVID.drain_new_amt, EVID.drain_dest_amt), np.nan)
EVID["eligible"] = (EVID.usable_shadow & (EVID.base_partner_amt >= ELIG_MIN_AMT) &
                    (EVID.base_partners >= ELIG_MIN_PARTNERS) & (EVID.post_months >= POST_MIN_M))
EVID["cont_else"] = np.where(EVID.eligible, sdiv(EVID.cont_else_amt, EVID.base_partner_amt), np.nan)
EVID["persist_any"] = np.where(EVID.eligible, sdiv(EVID.persist_amt, EVID.base_partner_amt), np.nan)

# shape features (§7 convergent validity) — none of them enters the typing rule
rels = list(range(REL_LO, REL_HI + 1))
BALW.columns = ["cust_pwr_id"] + [int(c) for c in BALW.columns[1:]]
EVID = EVID.merge(BALW, on="cust_pwr_id", how="left")
B = EVID[rels].apply(pd.to_numeric, errors="coerce").to_numpy(float)
ratio = B/normc[:, None]
def _first_below(th, start=None):
    ok = np.nan_to_num(ratio, nan=np.inf) < th
    if start is not None: ok &= np.arange(len(rels))[None, :] >= start[:, None]
    has = ok.any(axis=1)
    return np.where(has, np.argmax(ok, axis=1), -1)
i80 = _first_below(0.8); i20 = _first_below(0.2, np.maximum(i80, 0))
EVID["drain_months"] = np.where((i80 >= 0) & (i20 >= 0), (i20 - i80).astype(float), np.nan)
_post_idx = [rels.index(r) for r in range(POST_WIN[0], POST_WIN[1] + 1)]
EVID["rebound"] = np.nanmax(np.where(np.isfinite(ratio[:, _post_idx]), ratio[:, _post_idx], -np.inf), axis=1)
EVID.loc[~np.isfinite(EVID.rebound), "rebound"] = np.nan
EVID = EVID.drop(columns=rels)
EVID["ticket_ratio"] = np.log(sdiv(sdiv(EVID.drain_out_amt, EVID.drain_out_n), sdiv(EVID.base_out_amt, EVID.base_out_n)))
EVID["rcpt_ratio"] = np.log1p(EVID.drain_in_amt/nd) - np.log1p(EVID.base_in_amt/nb)
EVID["breadth"] = np.log1p(EVID.drain_n_cpty_out/nd) - np.log1p(EVID.base_n_cpty_out/nb)
EVID["lump"] = pd.to_numeric(EVID.drain_max_out, errors="coerce").fillna(0.0)/normc
EVID["wire_share"] = sdiv(EVID.drain_wire_out_amt, EVID.drain_out_amt)
save_frame(EVID, "v11_5b_evidence_per_client")

# ── evidence-to-date panel: at month t, recent (t-2..t) vs reference (t-8..t-5) ──
PANEL_P = hp("v11", "ev_panel")
if EVIDENCE_REBUILT or REBUILD_EVIDENCE or not hdfs_exists(PANEL_P + "/_SUCCESS"):
    months = to_sdf(pd.DataFrame({"m_idx": PAY_MONTHS}).astype("int64"))
    grid = TGT.crossJoin(months)
    V = ["out_amt", "out_n", "in_amt", "self_out_amt", "wire_out_amt", "int_sib_out_amt", "int_out_amt",
         "n_cpty_out", "max_out_txn"]
    g = grid.join(EVR["cm"].select("cust_pwr_id", "m_idx", *V), ["cust_pwr_id", "m_idx"], "left").na.fill(0.0)
    wR = Window.partitionBy("cust_pwr_id").orderBy("m_idx").rangeBetween(R_WIN[0], R_WIN[1])
    wF = Window.partitionBy("cust_pwr_id").orderBy("m_idx").rangeBetween(F_WIN[0], F_WIN[1])
    for v in V:
        agg = F.max if v == "max_out_txn" else F.sum
        g = g.withColumn(f"{v}_R", agg(v).over(wR)).withColumn(f"{v}_F", agg(v).over(wF))
    # new-institution dollars in the recent window, "new" relative to each t
    nr = (dst.join(first_seen, ["cust_pwr_id", "fin"])
          .withColumn("k", F.explode(F.array([F.lit(k) for k in range(0, R_WIN[1] - R_WIN[0] + 1)])))
          .withColumn("t", F.col("m_idx") + F.col("k")).filter(F.col("t") <= M_MAX)
          .groupBy("cust_pwr_id", F.col("t").alias("m_idx")).agg(
              F.sum("amt").alias("dest_R"),
              F.sum(F.when(F.col("fs") > F.col("t") + KNOWN_DEST_END, F.col("amt")).otherwise(0.0)).alias("new_R")))
    g = (g.join(nr, ["cust_pwr_id", "m_idx"], "left")
          .join(BAL.select("cust_pwr_id", "m_idx", "bal_med12"), ["cust_pwr_id", "m_idx"], "left"))
    nR, nFw = R_WIN[1] - R_WIN[0] + 1, F_WIN[1] - F_WIN[0] + 1
    bm = F.greatest(F.coalesce(F.col("bal_med12"), F.lit(0.0)), F.lit(DEFEND_MIN))
    def _r(a, b): return F.when(F.col(b) > 0, F.col(a)/F.col(b))
    P = g.select(
        "cust_pwr_id", "m_idx",
        _r("self_out_amt_R", "out_amt_R").alias("ev_self_share"),
        ((F.col("self_out_amt_R")/nR - F.col("self_out_amt_F")/nFw)/bm).alias("ev_self_excess"),
        F.when((F.col("m_idx") + KNOWN_DEST_END - M_MIN + 1 >= NEW_BLANK_M) & (F.col("dest_R") > 0),
               F.col("new_R")/F.col("dest_R")).alias("ev_new_share"),
        F.when(F.col("out_amt_R") + F.col("int_out_amt_R") > 0,
               F.col("int_sib_out_amt_R")/(F.col("out_amt_R") + F.col("int_out_amt_R"))).alias("ev_sib_share"),
        (F.log1p(F.col("in_amt_R")/nR) - F.log1p(F.col("in_amt_F")/nFw)).alias("ev_rcpt"),
        (F.log1p(F.col("out_amt_R")/nR) - F.log1p(F.col("out_amt_F")/nFw)).alias("ev_out"),
        F.when((F.col("out_n_R") > 0) & (F.col("out_n_F") > 0) & (F.col("out_amt_F") > 0) & (F.col("out_amt_R") > 0),
               F.log((F.col("out_amt_R")/F.col("out_n_R"))/(F.col("out_amt_F")/F.col("out_n_F")))).alias("ev_ticket"),
        _r("wire_out_amt_R", "out_amt_R").alias("ev_wire_share"),
        (F.col("max_out_txn_R")/bm).alias("ev_lump"),
        (F.log1p(F.col("n_cpty_out_R")/nR) - F.log1p(F.col("n_cpty_out_F")/nFw)).alias("ev_breadth"),
        F.when(F.col("m_idx") - M_MIN + F_WIN[0] >= 0, F.lit(1)).otherwise(F.lit(0)).alias("ev_full_ref"))
    P.write.mode("overwrite").parquet(PANEL_P)
EV_PANEL = spark.read.parquet(PANEL_P)
EV_COLS = [c for c in EV_PANEL.columns if c.startswith("ev_") and c != "ev_full_ref"]
def ev_encode(d):
    """ld_ev_* value (clipped, missing → 0) + md_ev_* missingness flag — the same
    encoding as the model's own features, so the two blocks are comparable."""
    out = d[["cust_pwr_id", "m_idx"]].copy()
    for c in EV_COLS:
        v = pd.to_numeric(d[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
        out[f"md_{c}"] = v.isna().astype(float)
        out[f"ld_{c}"] = v.clip(-10, 10).fillna(0.0)
    return out
EVX = [f"{p}_{c}" for c in EV_COLS for p in ("ld", "md")]

_g = EVID.groupby("grp").agg(clients=("cust_pwr_id", "size"), typeable=("typeable", "sum"),
                              own_key=("usable_self", "mean"), partner_key=("usable_shadow", "mean"),
                              eligible=("eligible", "sum"), normal_balance=("norm", "sum"))
disp(_g.reset_index().assign(own_key=_g.own_key.map(pctf).to_numpy(), partner_key=_g.partner_key.map(pctf).to_numpy(),
                             normal_balance=_g.normal_balance.map(usd).to_numpy()),
     title="5b &middot; <b>Who can be typed.</b> A = closed · B_only = drained, never closed · CTRL = stayers "
           "at pseudo-events drawn from the A event months. typeable = observable base window and normal "
           f"balance ≥ {usd(TYPE_MIN_NORM)} · eligible = relationship survival observable",
     save="v11_5b_population")
print(f"  §5b wall {(time.time()-t0)/60:,.1f} min · evidence-to-date features: {', '.join(EV_COLS)}")

In [ ]:
# =====================================================================
# 6 · DIRECT TYPING — thresholds from stayers, then the classes   [OUTPUT BLOCK 9]
# =====================================================================
t0 = time.time()
def _q(s, q):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    return float(s.quantile(q)) if len(s) else np.nan
def _hi(floor, v): return float(floor) if not np.isfinite(v) else float(max(floor, v))
def _lo(ceil, v): return float(ceil) if not np.isfinite(v) else float(min(ceil, v))

TYP = EVID.copy()
ST = TYP[(TYP.grp == "CTRL") & TYP.typeable]
STE = ST[ST.eligible]
THR = dict(self=_hi(FLOOR_SELF, _q(ST.e_self, STAYER_Q)),
           new=_hi(FLOOR_NEWBANK, _q(ST.e_new, STAYER_Q)),
           cont=_hi(FLOOR_CONT, _q(STE.cont_else, STAYER_Q)),
           sib=_hi(FLOOR_SIB, _q(ST.e_sib, STAYER_Q)),
           dead=_lo(CEIL_DEAD, _q(STE.persist_any, 1 - STAYER_Q)))
num = lambda c: pd.to_numeric(TYP[c], errors="coerce")
TYP["f_self"] = num("e_self").fillna(0) >= THR["self"]
TYP["f_cont"] = TYP.eligible & (num("cont_else").fillna(-1) >= THR["cont"])
TYP["f_new"] = num("e_new").fillna(-1) >= THR["new"]
TYP["f_sib"] = num("e_sib").fillna(0) >= THR["sib"]
TYP["dead"] = TYP.eligible & (num("persist_any").fillna(np.inf) <= THR["dead"])
TYP["disp_ev"] = TYP.f_self | TYP.f_cont

# ── the recall proxy: can the matcher see survival when it certainly exists? ──
_rp = TYP[(TYP.grp == "A") & TYP.typeable & TYP.eligible & (TYP.moved_self >= 0.5)]
RECALL_N = len(_rp)
RECALL = float((_rp.cont_else > 0).mean()) if RECALL_N else np.nan
def _wilson(p, n, z=1.96):
    if not n or not np.isfinite(p): return (np.nan, np.nan)
    d = 1 + z*z/n; c = (p + z*z/(2*n))/d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (c - h, c + h)
RECALL_CI = _wilson(RECALL, RECALL_N)
RECALL_OK = bool(RECALL_N >= RECALL_MIN_N and RECALL >= RECALL_MIN)

CLASSES = ["DISPLACEMENT", "MOVED_TO_NEW_BANK", "MIXED", "CONTRACTION", "LEANS_CONTRACTION",
           "INTERNAL", "NO_DIRECT_EVIDENCE", "NOT_TYPEABLE"]
TYP["type"] = np.select(
    [~TYP.typeable,
     TYP.f_sib & ~TYP.disp_ev,
     TYP.disp_ev & ~TYP.dead,
     TYP.disp_ev & TYP.dead,
     TYP.dead & TYP.f_new,
     TYP.dead,
     TYP.f_new],
    ["NOT_TYPEABLE", "INTERNAL", "DISPLACEMENT", "MIXED", "MIXED",
     "CONTRACTION" if RECALL_OK else "LEANS_CONTRACTION", "MOVED_TO_NEW_BANK"],
    "NO_DIRECT_EVIDENCE")
TYP["norm"] = pd.to_numeric(TYP.norm, errors="coerce").fillna(0.0)
save_frame(TYP, "v11_6_typed_clients")

# ── 6a · each indicator: threshold and how often it fires ─────────────
def _rate(g, f, base="typeable"):
    d = TYP[(TYP.grp == g) & TYP[base]]
    return float(d[f].mean()) if len(d) else np.nan
_ind = [("f_self", "e_self", "own-name transfers above own base, ÷ normal balance", FLOOR_SELF, "self", "typeable"),
        ("f_cont", "cont_else", "base partners seen trading with the client's name off-PNC", FLOOR_CONT, "cont", "eligible"),
        ("f_new", "e_new", "drain $ to institutions never used before rel_m −9", FLOOR_NEWBANK, "new", "typeable"),
        ("f_sib", "e_sib", "drain $ to own entities / siblings / same name, ÷ normal balance", FLOOR_SIB, "sib", "typeable"),
        ("dead", "persist_any", "base partners seen ANYWHERE after exit (fires when ≤ threshold)", CEIL_DEAD, "dead", "eligible"),
        ("disp_ev", "—", "displacement evidence: f_self or f_cont", np.nan, None, "typeable")]
_rows = []
for f, v, d, fl, k, base in _ind:
    ra, rc, rb = _rate("A", f, base), _rate("CTRL", f, base), _rate("B_only", f, base)
    _rows.append(dict(indicator=f, value=v, meaning=d, floor=fl,
                      stayer_quantile=(_q(STE[v] if base == "eligible" else ST[v], 1 - STAYER_Q if k == "dead" else STAYER_Q)
                                       if v != "—" else np.nan),
                      threshold=THR.get(k, np.nan) if k else np.nan, population=base,
                      fires_A=ra, fires_B_only=rb, fires_stayers=rc,
                      lift_A_vs_stayers=ra/rc if rc and np.isfinite(rc) and rc > 0 else np.nan))
IND = pd.DataFrame(_rows)
disp(IND.assign(**{c: IND[c].map(lambda v: f"{v:.3f}" if np.isfinite(v) else "—")
                   for c in ["floor", "stayer_quantile", "threshold", "lift_A_vs_stayers"]},
                **{c: IND[c].map(pctf) for c in ["fires_A", "fires_B_only", "fires_stayers"]}),
     title=f"6a &middot; <b>Indicators, calibrated so each fires for ≤ {1-STAYER_Q:.0%} of stayers.</b> "
           "threshold = max(floor, stayer 95th pct) — for <i>dead</i>, min(ceiling, stayer 5th pct). "
           "Rates are among typeable clients (eligible clients for the relationship indicators)",
     save="v11_6a_indicators")

# ── 6b · the classes ──────────────────────────────────────────────────
def _cls(g):
    d = TYP[TYP.grp == g]
    c = d.groupby("type").agg(clients=("cust_pwr_id", "size"), normal_balance=("norm", "sum")).reindex(CLASSES).fillna(0)
    t = d[d.typeable]
    c["share_of_typeable"] = c.clients/max(len(t), 1)
    c["share_of_typeable_$"] = c.normal_balance/max(float(t.norm.sum()), 1.0)
    c.loc["NOT_TYPEABLE", ["share_of_typeable", "share_of_typeable_$"]] = np.nan
    return c
CLS = {g: _cls(g) for g in ["A", "B_only", "CTRL"]}
_t = pd.concat({g: CLS[g][["clients", "share_of_typeable", "share_of_typeable_$"]] for g in CLS}, axis=1)
_t.columns = [f"{g} · {c}" for g, c in _t.columns]
disp(_t.reset_index(names="type").assign(
        **{c: _t[c].map(pctf).to_numpy() for c in _t.columns if "share" in c},
        **{c: _t[c].map(lambda v: f"{int(v):,}").to_numpy() for c in _t.columns if "clients" in c}),
     title=f"6b &middot; <b>Why they left — direct evidence.</b> Name-match recall proxy "
           f"{pctf(RECALL)} (n = {RECALL_N:,}) → relationships that stop are called "
           f"<b>{'CONTRACTION' if RECALL_OK else 'LEANS_CONTRACTION'}</b>. Stayers (CTRL) are typed at pseudo-events "
           "with the same rules: every class they land in is a false-positive rate",
     save="v11_6b_classes")
_a = CLS["A"]
disp(_a.reset_index(names="type").assign(normal_balance=_a.normal_balance.map(usd).to_numpy(),
                                         mean_balance=(_a.normal_balance/_a.clients.replace(0, np.nan)).map(usd).to_numpy(),
                                         share_of_typeable=_a.share_of_typeable.map(pctf).to_numpy(),
                                         **{"share_of_typeable_$": _a["share_of_typeable_$"].map(pctf).to_numpy()}),
     title="6c &middot; <b>A attriters: clients and normal balance (rel_m −12) by type</b>", save="v11_6c_A_dollars")

# by size band — does the mix change where the money is?
BANDS = [TYPE_MIN_NORM, 1e5, 1e6, 1e7, np.inf]
BLAB = ["$10k–100k", "$100k–1m", "$1m–10m", "$10m+"]
TA = TYP[(TYP.grp == "A") & TYP.typeable].copy()
TA["band"] = pd.cut(TA.norm, BANDS, labels=BLAB, right=False)
_bd = (TA.pivot_table(index="band", columns="type", values="norm", aggfunc="sum", observed=False)
       .reindex(columns=[c for c in CLASSES if c != "NOT_TYPEABLE"]).fillna(0.0))
_bd_share = _bd.div(_bd.sum(axis=1).replace(0, np.nan), axis=0)
_bd_n = TA.groupby("band", observed=False).size()
disp(_bd_share.assign(clients=_bd_n).reset_index().assign(
        **{c: _bd_share[c].map(pctf).to_numpy() for c in _bd_share.columns}),
     title="6d &middot; <b>A attriters: share of normal balance by type, within size band</b>",
     save="v11_6d_type_by_band")

if HAVE_MPL:
    _sh = pd.DataFrame({g: CLS[g].loc[[c for c in CLASSES if c != "NOT_TYPEABLE"], "share_of_typeable"]
                        for g in ["A", "B_only", "CTRL"]}).T.reset_index(names="group")
    bars(_sh, "group", [c for c in CLASSES if c != "NOT_TYPEABLE"], "v11_6b_class_mix",
         "6b · Why they left — class mix by group (typeable clients)",
         "stayers are typed at pseudo-events with the same rules: their bars are the false-positive floor",
         fmt=pctf, colors=TYPECOL, stacked=True)
    bars(_bd_share.reset_index().rename(columns={"band": "size band"}), "size band",
         [c for c in CLASSES if c != "NOT_TYPEABLE"], "v11_6d_type_by_band",
         "6d · A attriters — share of normal balance by type within size band", fmt=pctf,
         colors=TYPECOL, stacked=True)

ta, tc = TYP[(TYP.grp == "A") & TYP.typeable], TYP[(TYP.grp == "CTRL") & TYP.typeable]
_dn = float(ta.loc[ta.type.isin(["DISPLACEMENT", "CONTRACTION", "LEANS_CONTRACTION"]), "norm"].sum())
RES["disp_lift"] = float(ta.disp_ev.mean()/tc.disp_ev.mean()) if len(tc) and tc.disp_ev.mean() > 0 else np.nan
RES["disp_dollar_share"] = float(ta.loc[ta.type == "DISPLACEMENT", "norm"].sum()/_dn) if _dn > 0 else np.nan
RES["elig_share"] = float(ta.eligible.mean()) if len(ta) else np.nan
RES["recall_proxy"] = RECALL
RES["internal_share_A"] = float((ta.type == "INTERNAL").mean()) if len(ta) else np.nan
kv([("thresholds", " · ".join(f"{k} {v:.3f}" for k, v in THR.items())),
    ("A attriters typeable", f"{len(ta):,} of {int((TYP.grp == 'A').sum()):,} · normal balance {usd(ta.norm.sum())}"),
    ("…relationship survival observable (eligible)", pctf(RES["elig_share"])),
    ("recall proxy (eligible A with ≥ 50% of normal balance sent to own name)",
     f"{pctf(RECALL)}  [95% {pctf(RECALL_CI[0])} – {pctf(RECALL_CI[1])}] · n = {RECALL_N:,} · "
     f"{'gate passed' if RECALL_OK else 'gate NOT passed — dead relationships are only a lean'}"),
    ("displacement evidence, A vs stayers", f"{pctf(ta.disp_ev.mean())} vs {pctf(tc.disp_ev.mean())} · "
                                             f"lift {RES['disp_lift']:.2f}×"),
    ("displacement share of directly typed $ (DISP ÷ DISP + CONTR + LEANS)", pctf(RES["disp_dollar_share"])),
    ("internal moves among typeable A", pctf(RES["internal_share_A"])),
    ("wall", f"{time.time()-t0:,.0f}s")],
   title="6e &middot; <b>Typing summary</b>", save="v11_6e_summary")

In [ ]:
# =====================================================================
# 7 · IS THE TYPING BELIEVABLE? coverage, convergent validity, rebound [OUTPUT BLOCK 10]
# =====================================================================
# None of the shape features below enters the typing rule, so agreement with
# the predicted direction is independent evidence that the two groups differ
# in the way the story says. Disagreement is a finding, not a bug to tune away.
t0 = time.time()
DISP_SET, CONTR_SET = ["DISPLACEMENT"], ["CONTRACTION", "LEANS_CONTRACTION"]
TA = TYP[(TYP.grp == "A") & TYP.typeable].copy()
TA["band"] = pd.cut(TA.norm, BANDS, labels=BLAB, right=False)

# 7a · where can relationship survival be seen at all?
_cov = TA.groupby("band", observed=False).agg(
    clients=("cust_pwr_id", "size"), own_key=("usable_self", "mean"), partner_key=("usable_shadow", "mean"),
    base_partners_med=("base_partners", "median"), eligible=("eligible", "mean"),
    cont_else_pos=("cont_else", lambda s: float((s > 0).mean()) if s.notna().any() else np.nan),
    moved_self_50=("moved_self", lambda s: float((s >= 0.5).mean())))
disp(_cov.reset_index().assign(**{c: _cov[c].map(pctf).to_numpy()
                                  for c in ["own_key", "partner_key", "eligible", "cont_else_pos", "moved_self_50"]}),
     title="7a &middot; <b>Coverage of the evidence, by size band (typeable A attriters).</b> eligible = ≥ "
           f"{ELIG_MIN_PARTNERS} base-window PNC partners worth ≥ {usd(ELIG_MIN_AMT)}, a partner-matchable name, "
           f"≥ {POST_MIN_M} post months observed · cont_else_pos among eligible · moved_self_50 = sent ≥ half the "
           "normal balance to its own name elsewhere", save="v11_7a_coverage")

# 7b · convergent validity: displacement vs contraction on features the rule never saw
VAL = TA[TA.type.isin(DISP_SET + CONTR_SET)].copy()
VAL["yd"] = VAL.type.isin(DISP_SET).astype(int)
SHAPES = [("drain_months", -1, "months from 80% to 20% of normal balance", "displacement drains faster"),
          ("lump", +1, "largest single payment in the drain ÷ normal balance", "one big transfer = a move"),
          ("wire_share", +1, "wire share of drain outflow", "balances move by wire"),
          ("rcpt_ratio", +1, "receipts, drain vs base (log)", "a trading business keeps being paid"),
          ("breadth", +1, "counterparties paid, drain vs base (log)", "suppliers still paid until the move"),
          ("ticket_ratio", +1, "average payment, drain vs base (log)", "fewer but larger payments")]
def _boot_auc(y, s, n=1000, seed=SEED):
    y = np.asarray(y, float); s = np.asarray(s, float); ok = np.isfinite(s); y, s = y[ok], s[ok]
    if len(y) < 10 or y.min() == y.max(): return np.nan, np.nan, np.nan, int(len(y))
    a = auc(y, s); r = np.random.default_rng(seed); b = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if y[i].min() != y[i].max(): b.append(auc(y[i], s[i]))
    lo, hi = np.percentile(b, [2.5, 97.5]) if b else (np.nan, np.nan)
    return a, lo, hi, int(len(y))
_rows = []
for c, sgn, d, why in SHAPES:
    a, lo, hi, n = _boot_auc(VAL.yd, sgn*pd.to_numeric(VAL[c], errors="coerce"))
    _rows.append(dict(feature=c, meaning=d, predicted=why, n=n, auc=a, ci_lo=lo, ci_hi=hi,
                      median_displacement=pd.to_numeric(VAL.loc[VAL.yd == 1, c], errors="coerce").median(),
                      median_contraction=pd.to_numeric(VAL.loc[VAL.yd == 0, c], errors="coerce").median(),
                      verdict=("as predicted" if np.isfinite(lo) and lo > 0.5 else
                               "opposite" if np.isfinite(hi) and hi < 0.5 else "no separation")))
CONV = pd.DataFrame(_rows)
RES["drain_auc"] = float(CONV.loc[CONV.feature == "drain_months", "auc"].iloc[0])
disp(CONV.assign(**{c: CONV[c].map(lambda v: f"{v:.3f}" if np.isfinite(v) else "—")
                    for c in ["auc", "ci_lo", "ci_hi", "median_displacement", "median_contraction"]}),
     title=f"7b &middot; <b>Convergent validity.</b> {int(VAL.yd.sum()):,} displacement vs "
           f"{int((1 - VAL.yd).sum()):,} contraction (incl. leans). AUC oriented so &gt; 0.5 = the predicted "
           "direction; 95% client bootstrap", save="v11_7b_convergent")
barh_err([f"{r.feature}: {r.predicted}" for r in CONV.itertuples()], CONV.auc - 0.5,
         CONV.ci_lo - 0.5, CONV.ci_hi - 0.5, [ACC if v > 0 else ACC2 for v in CONV.auc.fillna(0.5) - 0.5],
         "v11_7b_convergent", "7b · Do the typed groups differ the way the story says?",
         "AUC − 0.5, displacement vs contraction, oriented to the prediction · 95% bootstrap",
         xlab="AUC − 0.5 (right = as predicted)")

# 7c · B-only clients (drained, never closed): does the money come back, by type?
TB = TYP[(TYP.grp == "B_only") & TYP.typeable].copy()
_rb = TB.groupby("type").agg(clients=("cust_pwr_id", "size"), normal_balance=("norm", "sum"),
                             rebound_median=("rebound", "median"),
                             back_half=("rebound", lambda s: float((s >= 0.5).mean()) if s.notna().any() else np.nan),
                             gone=("rebound", lambda s: float((s < 0.1).mean()) if s.notna().any() else np.nan))
disp(_rb.reindex([c for c in CLASSES if c in _rb.index]).reset_index().assign(
        normal_balance=lambda d: d.normal_balance.map(usd), rebound_median=lambda d: d.rebound_median.map(f4),
        back_half=lambda d: d.back_half.map(pctf), gone=lambda d: d.gone.map(pctf)),
     title="7c &middot; <b>B-only clients: does the balance come back, by type?</b> rebound = highest post-event "
           "balance ÷ normal balance. Expect displacement ≈ gone; parked cash shows up as rebound under "
           "INTERNAL or NO_DIRECT_EVIDENCE", save="v11_7c_b_rebound")

# 7d · the sample RMs are asked to confirm — the only real validation there is
_s = TA[TA.type != "NOT_TYPEABLE"].copy()
_s["_h"] = pd.util.hash_pandas_object(_s.cust_pwr_id + f"|rm|{SEED}", index=False).to_numpy()
SAMPLE = (_s.sort_values(["type", "_h"]).groupby("type").head(RM_SAMPLE_PER_TYPE)
          [["cust_pwr_id", "type", "e", "norm", "e_self", "moved_self", "cont_else", "persist_any", "e_new",
            "e_sib", "base_partners", "drain_months", "lump", "rcpt_ratio"]])
SAMPLE["event_month"] = SAMPLE.e.map(lambda m: f"{(m - 1)//12}-{(m - 1) % 12 + 1:02d}")
SAMPLE["rm_verdict"] = ""; SAMPLE["rm_notes"] = ""
SAMPLE.to_csv(OUT_DIR / "v11_7d_rm_validation_sample_FULL_IDS.csv", index=False)
disp(SAMPLE.groupby("type").agg(clients=("cust_pwr_id", "size"), normal_balance=("norm", "sum"))
     .reindex([c for c in CLASSES if c in set(SAMPLE.type)]).reset_index()
     .assign(normal_balance=lambda d: d.normal_balance.map(usd)),
     title=f"7d &middot; <b>RM validation sample</b> — {RM_SAMPLE_PER_TYPE} per type, deterministic. Full ids "
           "and evidence in v11_7d_rm_validation_sample_FULL_IDS.csv with blank rm_verdict / rm_notes columns")
print(f"  §7 wall {time.time()-t0:,.0f}s")

## 8 · Can the type be predicted while the money is still here?

The direct evidence in §6 is mostly visible **around and after** the exit — self-transfers peak in the last
months, relationship survival is measured in the post window. An RM needs the answer at the alert, which
arrives rel_m −6 … −1. So the directly typed A attriters become labels (**displacement = 1**, contraction and
leans-contraction = 0; mixed, new-bank-only and no-evidence clients are left out rather than guessed), and a
second model is fitted on their **alert-time rows** (money still here, rel_m −6 … −1) from three blocks:

| Block | Columns | Question |
|---|---|---|
| `model` | the 148 all-features columns | does the attrition model's own view already carry the type? |
| `evidence` | evidence-to-date: own-name share and excess, new-institution share, sibling share, receipts, outflow, ticket, wire share, largest payment, breadth — recent 3 months vs months −8 … −5 | does the partial evidence visible *before* exit carry it? |
| `both` | union | the model used in §9 (fixed in advance, not chosen on the result) |

Grouped 5-fold cross-validation **by client** (a client's six rows never straddle train and test); each client
weighted equally regardless of how many alert-time rows it has. A second, temporal split trains on earlier
events and tests on later ones. The output is **p(displacement | leaving)** — a conditional probability, only
meaningful next to p(leave), which is how §9 uses it.

In [ ]:
# =====================================================================
# 8 · p(displacement | leaving) AT ALERT TIME                   [OUTPUT BLOCK 11]
# =====================================================================
t0 = time.time()
EV_DESC = {"ev_self_share": "share of recent outflow sent to its own name elsewhere",
           "ev_self_excess": "own-name transfers, recent vs earlier, ÷ normal balance",
           "ev_new_share": "share of recent institution $ going to new institutions",
           "ev_sib_share": "share of recent outflow to own entities / siblings",
           "ev_rcpt": "receipts, recent vs earlier", "ev_out": "outflow, recent vs earlier",
           "ev_ticket": "average payment, recent vs earlier", "ev_wire_share": "wire share of recent outflow",
           "ev_lump": "largest recent payment ÷ normal balance",
           "ev_breadth": "counterparties paid, recent vs earlier"}
def describe_col(c):
    s = sig_of(c); d = EV_DESC.get(s) or describe(s)
    return d + (" — missing flag" if c.startswith("md_") else "")

LBL = TYP[(TYP.grp == "A") & TYP.type.isin(DISP_SET + CONTR_SET)][["cust_pwr_id", "e", "type"]].copy()
LBL["yd"] = LBL.type.isin(DISP_SET).astype(float)
n_d, n_c = int(LBL.yd.sum()), int((1 - LBL.yd).sum())
TYPE_OK = min(n_d, n_c) >= MIN_TYPE_N
TYPE_SPEC, FOLD_SPEC, CV_FOLD, TR8 = None, {}, {}, None
BLOCKS = {"model": COLS, "evidence": EVX, "both": COLS + EVX}
REL8 = list(range(-6, 0))

if TYPE_OK:
    _lb = to_sdf(LBL[["cust_pwr_id", "e"]].astype({"e": "int64"}))
    TR8 = collect_pd(RS.withColumn("cust_pwr_id", F.col("cust_pwr_id").cast("string"))
                     .join(F.broadcast(_lb), "cust_pwr_id")
                     .filter((F.col("m_idx") - F.col("e")).between(REL8[0], REL8[-1])), "alert-time rows, typed clients")
    TR8 = prep(TR8)
    TR8 = TR8[TR8.defend].merge(LBL[["cust_pwr_id", "type", "yd"]], on="cust_pwr_id")
    TR8["rel"] = TR8.m_idx - pd.to_numeric(TR8.e, errors="coerce").astype(int)
    _k = to_sdf(TR8[["cust_pwr_id", "m_idx"]].astype({"m_idx": "int64"}))
    TR8 = TR8.merge(ev_encode(collect_pd(EV_PANEL.join(F.broadcast(_k), ["cust_pwr_id", "m_idx"]),
                                         "evidence-to-date rows")), on=["cust_pwr_id", "m_idx"], how="left")
    for c in EVX: TR8[c] = TR8[c].fillna(1.0 if c.startswith("md_") else 0.0)
    TR8["y"] = TR8.yd
    TR8["sw"] = 1.0/TR8.groupby("cust_pwr_id").cust_pwr_id.transform("size")   # each client counts once
    CV_FOLD = dict(zip(LBL.cust_pwr_id, pd.util.hash_pandas_object(LBL.cust_pwr_id + f"|type|{SEED}", index=False)
                       .to_numpy() % TYPE_FOLDS))
    TR8["fold"] = TR8.cust_pwr_id.map(CV_FOLD).astype(int)
    TYPE_OK = min(TR8.groupby("cust_pwr_id").yd.first().value_counts().reindex([0.0, 1.0]).fillna(0)) >= MIN_TYPE_N

if TYPE_OK:
    OOF = {b: np.full(len(TR8), np.nan) for b in BLOCKS}
    for k in range(TYPE_FOLDS):
        trm = (TR8.fold != k).to_numpy()
        for b, cols in BLOCKS.items():
            sp = fit_spec(TR8[trm], cols)
            assert sp is not None and sp["converged"], f"type model {b}, fold {k}: no convergence"
            OOF[b][~trm] = predict_p(sp, TR8[~trm])
            if b == "both": FOLD_SPEC[k] = sp
    for b in BLOCKS: TR8[f"oof_{b}"] = OOF[b]
    def _cauc(b):
        g = TR8.groupby("cust_pwr_id").agg(y=("yd", "first"), p=(f"oof_{b}", "mean"))
        return auc(g.y, g.p)
    _rows = []
    for b in BLOCKS:
        r = dict(block=b, columns=len(BLOCKS[b]), rows=len(TR8), clients=TR8.cust_pwr_id.nunique(),
                 auc_rows=auc(TR8.yd, TR8[f"oof_{b}"]), auc_clients=_cauc(b))
        for rm in REL8:
            m = (TR8.rel == rm).to_numpy()
            r[f"auc_rel{rm}"] = auc(TR8.yd[m], TR8[f"oof_{b}"][m])
        _rows.append(r)
    T8 = pd.DataFrame(_rows)
    # temporal split: earlier events train, later events test
    e_cut = float(LBL.e.median())
    _tr, _te = TR8[TR8.e <= e_cut], TR8[TR8.e > e_cut]
    T8["auc_temporal"] = [auc(_te.yd, predict_p(fit_spec(_tr, BLOCKS[b]), _te))
                          if _tr.yd.nunique() == 2 and _te.yd.nunique() == 2 else np.nan for b in BLOCKS]
    RES["type_auc"] = float(T8.loc[T8.block == "both", "auc_rows"].iloc[0])
    disp(T8.assign(**{c: T8[c].map(lambda v: f"{v:.3f}" if np.isfinite(v) else "—")
                      for c in T8.columns if c.startswith("auc")}),
         title=f"8a &middot; <b>p(displacement | leaving), grouped {TYPE_FOLDS}-fold CV by client.</b> "
               f"{n_d:,} displacement vs {n_c:,} contraction clients · base rate {n_d/(n_d+n_c):.1%} · "
               f"rows = money-still-here client-months at rel_m −6…−1 · temporal = trained on events ≤ "
               f"m_idx {e_cut:.0f}, tested after", save="v11_8a_type_model")
    if HAVE_MPL:
        fig, ax = plt.subplots(figsize=(8, 4))
        for b, col in zip(BLOCKS, [ACC2, WARN, ACC]):
            ax.plot(REL8, T8.loc[T8.block == b, [f"auc_rel{r}" for r in REL8]].to_numpy()[0], marker="o",
                    color=col, label=b)
        ax.axhline(0.5, color=INK, lw=.8, ls=":"); ax.axhline(0.7, color=GREY, lw=.8, ls="--")
        ax.set_xlabel("months before exit (alert month)"); ax.set_ylabel("AUC, displacement vs contraction")
        ax.legend(frameon=False)
        _save(fig, "v11_8a_type_auc_by_month", "8a · How early can the type be told apart?",
              "grouped CV by client · money still here · dashed line = the 0.70 pre-registered bar")
    TYPE_SPEC = fit_spec(TR8, BLOCKS["both"])
    assert TYPE_SPEC["converged"], "final type model did not converge"
    _co = pd.DataFrame({"column": TYPE_SPEC["cols"], "b_std": TYPE_SPEC["b_std"]})
    _co["abs"] = _co.b_std.abs(); _co = _co.sort_values("abs", ascending=False).head(20)
    _co["reads_as"] = np.where(_co.b_std > 0, "higher → displacement", "higher → contraction")
    _co["meaning"] = _co.column.map(describe_col)
    _co["block"] = np.where(_co.column.str.contains("_ev_"), "evidence", "model")
    disp(_co[["column", "meaning", "block", "b_std", "reads_as"]].assign(b_std=_co.b_std.map(lambda v: f"{v:+.3f}")),
         title="8b &middot; <b>What separates the two at alert time</b> — the 20 largest standardised "
               "coefficients of the final <code>both</code> model (ridge: a coefficient is a direction, not an importance)",
         save="v11_8b_type_coefficients")
    TR8[["cust_pwr_id", "m_idx", "rel", "type", "yd", "fold"] + [f"oof_{b}" for b in BLOCKS]].to_csv(
        OUT_DIR / "v11_8_type_oof_FULL_IDS.csv", index=False)
else:
    RES["type_auc"] = np.nan
    print(f"  §8 skipped: {n_d} displacement / {n_c} contraction clients — fewer than MIN_TYPE_N = {MIN_TYPE_N} "
          "in a class. The direct typing (§6) still stands; the alert-time model needs more labels.")
print(f"  §8 wall {time.time()-t0:,.0f}s")

In [ ]:
# =====================================================================
# 9 · THE RM VIEW — p(leave) × p(displacement | leaving) → a play [OUTPUT BLOCK 12]
# =====================================================================
# Every first alert of the recommended queue gets p_disp from the §8 `both`
# model. Clients whose own history trained that model are scored by the fold
# model that never saw them, so no alert is scored in-sample.
#   VERIFY_INTERNAL  recent outflow mostly to own entities / siblings  → check for a re-key first
#   COMPETE          p_disp ≥ 0.65 → price, ECR, service, product
#   SUPPORT          p_disp ≤ 0.35 → credit, working capital, restructuring, patience
#   DISCOVER         in between    → ask before offering either
t0 = time.time()
PLAYS = ["VERIFY_INTERNAL", "COMPETE", "DISCOVER", "SUPPORT"]
PLAYCOL = {"VERIFY_INTERNAL": GOOD, "COMPETE": ACC, "DISCOVER": WARN, "SUPPORT": ACC2}
QA = Q_FIRST[["cust_pwr_id", "m_idx", "y", "p_full", "bar_now", "bar_med12", "bar_cap"]].copy()
_x = []
for T in ORIG:
    q = QA[QA.m_idx == T]
    if len(q): _x.append(q.merge(TE[T][["cust_pwr_id"] + COLS].drop_duplicates("cust_pwr_id"), on="cust_pwr_id", how="left"))
QX = pd.concat(_x, ignore_index=True)
_k = to_sdf(QX[["cust_pwr_id", "m_idx"]].astype({"m_idx": "int64"}))
_ev = collect_pd(EV_PANEL.join(F.broadcast(_k), ["cust_pwr_id", "m_idx"]), "evidence-to-date at alert")
QX = QX.merge(ev_encode(_ev), on=["cust_pwr_id", "m_idx"], how="left")
for c in EVX: QX[c] = QX[c].fillna(1.0 if c.startswith("md_") else 0.0)
QX = QX.merge(_ev[["cust_pwr_id", "m_idx", "ev_sib_share"]], on=["cust_pwr_id", "m_idx"], how="left")

if TYPE_SPEC is not None:
    QX["p_disp"] = predict_p(TYPE_SPEC, QX)
    fk = QX.cust_pwr_id.map(CV_FOLD)
    for k, sp in FOLD_SPEC.items():
        m = (fk == k).to_numpy()
        if m.any(): QX.loc[m, "p_disp"] = predict_p(sp, QX[m])
    QX["scored_by"] = np.where(fk.notna(), "fold model (client was a label)", "final model")
else:
    QX["p_disp"] = np.nan; QX["scored_by"] = "no type model"
sib = pd.to_numeric(QX.ev_sib_share, errors="coerce").fillna(0) >= FLOOR_SIB
QX["play"] = np.select([sib, QX.p_disp >= P_COMPETE, QX.p_disp <= P_SUPPORT], PLAYS[:2] + ["SUPPORT"], "DISCOVER")
QX.loc[QX.p_disp.isna() & ~sib, "play"] = "DISCOVER"

# 9a · the queue by play
_pl = (QX.groupby("play").agg(alerts=("cust_pwr_id", "size"), departing=("y", "sum"),
                              capped_at_alert=("bar_cap", "sum"),
                              capped_departing=("bar_cap", lambda s: float(s[QX.loc[s.index, "y"] == 1].sum())),
                              median_p_disp=("p_disp", "median"))
       .reindex(PLAYS).fillna(0))
_pl["alert_share"] = _pl.alerts/max(_pl.alerts.sum(), 1)
_pl["precision"] = _pl.departing/_pl.alerts.replace(0, np.nan)
_pl["share_of_departing_$"] = _pl.capped_departing/max(_pl.capped_departing.sum(), 1.0)
disp(_pl.reset_index().assign(capped_at_alert=_pl.capped_at_alert.map(usd).to_numpy(),
                              capped_departing=_pl.capped_departing.map(usd).to_numpy(),
                              median_p_disp=_pl.median_p_disp.map(lambda v: f"{v:.2f}").to_numpy(),
                              alert_share=_pl.alert_share.map(pctf).to_numpy(),
                              precision=_pl.precision.map(pctf).to_numpy(),
                              **{"share_of_departing_$": _pl["share_of_departing_$"].map(pctf).to_numpy()}),
     title=f"9a &middot; <b>The recommended queue, split by play.</b> {len(QX):,} first alerts over "
           f"{len(ORIG)} months (K = {QUEUE_K:,}, α = {ALPHA}). capped_departing = capped balance of the alerts "
           "that did leave — the dollars each play would be defending", save="v11_9a_plays")
bars(_pl.reset_index().assign(d=_pl.capped_departing.to_numpy()), "play", ["d"], "v11_9a_plays_dollars",
     "9a · Capped dollars of departing clients reached, by play", "first alerts · money still here · 7 test months",
     colors={"d": ACC})

# 9b · on alerts whose client was later typed directly: does the play agree?
AG = QX.merge(TYP[["cust_pwr_id", "type"]], on="cust_pwr_id", how="left")
AG = AG[AG.y == 1]
_ct = pd.crosstab(AG.play, AG.type.fillna("NOT_LABELLED")).reindex(index=PLAYS).fillna(0).astype(int)
_ct = _ct[[c for c in CLASSES + ["NOT_LABELLED"] if c in _ct.columns]]
disp(_ct.reset_index(),
     title="9b &middot; <b>Departing alerts: play at alert time vs direct type after exit.</b> The fold-model "
           "scores make the COMPETE×DISPLACEMENT and SUPPORT×CONTRACTION cells an honest agreement check",
     save="v11_9b_agreement")
_c = AG[AG.type.isin(DISP_SET + CONTR_SET) & AG.play.isin(["COMPETE", "SUPPORT"])]
AGREE = float(((_c.play == "COMPETE") == _c.type.isin(DISP_SET)).mean()) if len(_c) else np.nan

# 9c · the latest month, as an RM would receive it
L9 = QX[QX.m_idx == max(ORIG)].merge(REASONS[["cust_pwr_id", "m_idx", "reason_1", "reason_2", "reason_3"]],
                                     on=["cust_pwr_id", "m_idx"], how="left")
L9["score"] = qscore(L9.p_full, L9.bar_med12)
L9 = L9.sort_values("score", ascending=False)
L9[["cust_pwr_id", "m_idx", "p_full", "p_disp", "play", "bar_med12", "bar_now", "reason_1", "reason_2",
    "reason_3", "scored_by", "y"]].to_csv(OUT_DIR / "v11_9c_rm_list_with_play_FULL_IDS.csv", index=False)
_h = L9.head(25)
disp(_h.assign(client=_h.cust_pwr_id.map(cid), p_full=_h.p_full.map(pctf), p_disp=_h.p_disp.map(lambda v: pctf(v, 0)),
               bar_med12=_h.bar_med12.map(usd))[["client", "p_full", "p_disp", "play", "bar_med12", "reason_1",
                                                  "reason_2", "reason_3", "y"]],
     title=f"9c &middot; <b>Month {max(ORIG)}: top 25 new alerts — why they're on the list and what to lead with.</b> "
           "p_disp = p(displacement | leaving). y known only in back-test")
RES["play_agreement"] = AGREE
kv([("alerts scored for type", f"{int(QX.p_disp.notna().sum()):,} of {len(QX):,}"),
    ("…by a fold model (client was a training label)", f"{int((QX.scored_by.str.startswith('fold')).sum()):,}"),
    ("plays", " · ".join(f"{p} {int(_pl.loc[p, 'alerts']):,}" for p in PLAYS)),
    ("agreement on confident calls (COMPETE↔DISPLACEMENT, SUPPORT↔CONTRACTION)",
     f"{pctf(AGREE)} of {len(_c):,} departing, directly typed alerts"),
    ("wall", f"{time.time()-t0:,.0f}s")], title="9d &middot; <b>RM view summary</b>", save="v11_9d_summary")

In [ ]:
# =====================================================================
# 10 · SCORECARD — the pre-registered predictions, scored mechanically [OUTPUT BLOCK 13]
# 11 · CHART INDEX
# =====================================================================
def _chk(key, test, fmt=lambda v: f"{v:.4f}"):
    v = RES.get(key, np.nan)
    try: ok = bool(test(v)) if (isinstance(v, str) or np.isfinite(v)) else None
    except TypeError: ok = None
    shown = v if isinstance(v, str) else (fmt(v) if np.isfinite(v) else "—")
    return shown, ("✅ held" if ok else "❌ failed" if ok is not None else "— not measured")
PRED = [
    ("P1", "refit AUC within ±0.003 of 0.7483", "~85%", "auc_refit", lambda v: abs(v - REF_AUC_DEFEND) <= 0.003, None),
    ("P2", "top-10 signals within 0.010 AUC of all 74 (late folds)", "~60%", "top10_gap", lambda v: v <= 0.010, None),
    ("P3", "payment_v7 is the costliest family to drop (capped $)", "~75%", "top_family", lambda v: v == "payment_v7", None),
    ("P4", "≥ 1/3 of signals add nothing uniquely", "~65%", "share_no_unique", lambda v: v >= 1/3, pctf),
    ("P5", "standalone vs unique rank Spearman < 0.6", "~60%", "rank_spearman", lambda v: v < 0.6, None),
    ("P6", "displacement evidence ≥ 2× stayers", "~70%", "disp_lift", lambda v: v >= 2.0, lambda v: f"{v:.2f}×"),
    ("P7", "displacement ≥ 60% of directly typed $", "~60%", "disp_dollar_share", lambda v: v >= 0.60, pctf),
    ("P8", "relationship survival observable for ≥ 40%", "~50%", "elig_share", lambda v: v >= 0.40, pctf),
    ("P9", "name-matching recall proxy ≥ 50%", "~50%", "recall_proxy", lambda v: v >= 0.50, pctf),
    ("P10", "drain speed separates the types, AUC ≥ 0.60", "~55%", "drain_auc", lambda v: v >= 0.60, None),
    ("P11", "p(displacement | leaving) grouped-CV AUC ≥ 0.70", "~50%", "type_auc", lambda v: v >= 0.70, None),
    ("P12", "internal moves < 5% of typeable A", "~70%", "internal_share_A", lambda v: v < 0.05, pctf)]
_rows = []
for pid, txt, conf, key, test, fmt in PRED:
    shown, verdict = _chk(key, test, fmt or (lambda v: f"{v:.4f}"))
    _rows.append(dict(id=pid, prediction=txt, confidence=conf, measured=shown, verdict=verdict))
SCORE = pd.DataFrame(_rows)
_held = int(SCORE.verdict.str.startswith("✅").sum()); _meas = int((~SCORE.verdict.str.startswith("—")).sum())
disp(SCORE, title=f"10 &middot; <b>Pre-registered predictions: {_held} of {_meas} measured held.</b> "
                  "Stated before any v11 number existed; a failed prediction is a finding", n=len(SCORE),
     save="v11_10_scorecard")
pd.DataFrame([dict(key=k, value=v) for k, v in RES.items()]).to_csv(OUT_DIR / "v11_10_results.csv", index=False)
CHARTS = pd.DataFrame(SAVED)
disp(CHARTS, title=f"11 &middot; <b>Chart index</b> — {len(CHARTS)} charts saved to {OUT_DIR} (never displayed)",
     n=len(CHARTS), save="v11_11_chart_index")

## Reading this run

**Question 1 — what drives the score.** Read 3b top-down, but trust the *unique* column (`d_drop`) over the
standalone one: a signal that is strong alone and costs nothing to remove is a story, not a requirement. The
family table in 3c is the one to quote in dollars. If the reduced models hold (P2), the production spec can be
cut to the top 10–20 signals with the cost stated in capped dollars, not just AUC. The reason codes in §4
explain the *model*: "counterparties paid: share of standing relationships that stopped — higher than
typical" means that signal pushed this client up the list, not that it caused the departure.

**Question 2 — why they leave.** Read 6a before 6b: every class is only as good as its indicators' false
positive rate on stayers. Then read the gates in order:

1. **Recall proxy (6e).** Below 50%, a relationship that "stopped" is too often a name the matcher missed —
   the contraction call is a lean, and the notebook says so in the class name.
2. **Eligibility (7a).** The survival test only covers clients whose trading partners also bank at PNC. If
   that is mostly the smaller bands, the dollar mix in 6c is a statement about them.
3. **Convergent validity (7b).** Displacement and contraction should differ on features the rule never used
   — drain speed, lump size, receipts. "No separation" on all of them would mean the classes are noise.
4. **Alert-time model (8a).** The typing is only useful to an RM if it is visible before the money leaves.
   The AUC by month says how early.

**What this is not.** The typing is a hypothesis about *why* built from payment behaviour. It becomes a
finding when RMs confirm the §7d sample, and a product when win/loss outcomes are recorded against the play
they were given. Until then COMPETE / SUPPORT is a conversation opener, not an offer — and any routing of
credit offers by this score needs the fair-lending review first.

**Send back:** photos of blocks 2b, 3a, 3b (top 25 rows), 3c (both tables), 4a, 6a, 6b, 6e, 7a, 7b, 8a, 9a,
9b and 10, plus `v11_7d_rm_validation_sample_FULL_IDS.csv` to whoever can arrange the RM review.